Regresión Lineal Multivariante

In [8]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas Académicas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 22
plt.rcParams['axes.labelsize'] = 18
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 16
plt.rcParams['lines.linewidth'] = 2.5

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

# --- PARÁMETROS MODIFICADOS ---
N_LAGS_AQI = 120  # Ampliado a ~5 días
N_LAGS_EXO = 120  # Ampliado a ~5 días

PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_CV_SPLITS = 5

# --- Parámetros de Ploteo ---
N_PLOT_EXAMPLES = 1
HISTORY_LEN_FOR_PLOT = 300

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs')
ORIGINAL_DATA_FILE = os.path.join(ORIGINAL_DATA_DIR, 'Air_Quality.csv')
SIMULATED_NANS_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'regresionLinealMultipleLags_full_plots'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_CSVS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'predicciones_csv')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')

os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_CSVS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def create_multivariate_lagged_features(df_full_city_data, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon):
    df_copy = df_full_city_data.copy()
    features_list = []
    for i in range(horizon, n_lags_target + horizon):
        features_list.append(df_copy[target_col].shift(i).rename(f'{target_col}_lag{i}'))
    for exo_col in exo_cols:
        if exo_col == target_col: continue
        for i in range(horizon, n_lags_exo + horizon):
            features_list.append(df_copy[exo_col].shift(i).rename(f'{exo_col}_lag{i}'))
    for tf_col in time_features_cols:
        if tf_col in df_copy.columns:
            features_list.append(df_copy[tf_col])
    df_features = pd.concat(features_list, axis=1)
    return df_features

# --- FUNCIÓN DE PLOTEO CORREGIDA PARA ASEGURAR HISTORIA IMPUTADA ---
def generate_multi_step_forecasting_plots(df_imputed_data, df_ground_truth_data, model, X_all_features, original_nan_indices, city, scenario, method, rep_id, horizon, output_dir, n_examples=1, history_len=300):
    max_lags = max(N_LAGS_AQI, N_LAGS_EXO)
    
    all_possible_points = range(max_lags + history_len, len(df_imputed_data) - horizon - 1)
    
    # --- LÓGICA DE FILTRADO CORREGIDA Y REFORZADA ---
    valid_split_points = []
    if not original_nan_indices: # Caso Baseline: no se requiere historia imputada
        valid_split_points = list(all_possible_points)
    else:
        for p in all_possible_points:
            future_window = range(p, p + horizon)
            history_window = range(p - max_lags, p)
            # Condición 1: El futuro DEBE estar limpio
            is_future_clean = not any(idx in original_nan_indices for idx in future_window)
            # Condición 2: La historia DEBE contener imputaciones
            is_history_dirty = any(idx in original_nan_indices for idx in history_window)
            if is_future_clean and is_history_dirty:
                valid_split_points.append(p)

    if len(valid_split_points) < n_examples:
        print(f"      Not enough valid windows (with imputed history and real future) to generate {n_examples} plots for h={horizon}.")
        return

    example_indices = random.sample(valid_split_points, n_examples)

    for i, split_point in enumerate(example_indices):
        history = df_imputed_data.iloc[:split_point]
        ground_truth_df = df_ground_truth_data.iloc[split_point : split_point + horizon]
        X_forecast_features = X_all_features.loc[ground_truth_df.index]
        if X_forecast_features.empty or X_forecast_features.isnull().values.any(): continue
        forecast_values = model.predict(X_forecast_features)
        lower_bound = history[TARGET_VARIABLE].quantile(0.01); upper_bound = history[TARGET_VARIABLE].quantile(0.99)
        padding = (upper_bound - lower_bound) * 0.1
        clipped_forecast_values = np.clip(forecast_values, lower_bound - padding, upper_bound + padding)
        forecast_df = pd.DataFrame({'Forecast': clipped_forecast_values}, index=ground_truth_df.index)

        plt.figure()
        plot_history = history.iloc[-history_len:]; plt.plot(plot_history.index, plot_history[TARGET_VARIABLE], label='Observed History', color='royalblue')
        plt.plot(ground_truth_df.index, ground_truth_df[TARGET_VARIABLE], label='Ground Truth', color='red')
        plt.plot(forecast_df.index, forecast_df['Forecast'], label='Forecast', color='green', linestyle='--')
        plt.axvline(x=split_point - 0.5, color='black', linestyle='--', linewidth=1.5, label='Forecast Start')
        plt.title(f'Linear Regressor Forecast in {city} for {horizon}-Hour Horizon\nImputation: {method} | Scenario: {scenario}')
        plt.xlabel('Time Step'); plt.ylabel(f'{TARGET_VARIABLE} Value'); plt.legend(loc='upper left')
        plt.grid(True, which='both', linestyle='--', linewidth=0.5); plt.tight_layout()
        scenario_safe_for_path = scenario.replace('/', '_').replace('\\', '_'); plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe_for_path); os.makedirs(plot_save_dir, exist_ok=True)
        plot_filename = f'forecast_example_{city.replace(" ", "_")}_{scenario_safe_for_path}_{rep_id}_h{horizon}.png'; plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150); plt.close()

# --- FUNCIÓN DE EVALUACIÓN CORREGIDA PARA FORZAR HISTORIA IMPUTADA ---
def train_predict_eval_multi_horizon(df_city_data_imputed, df_city_data_ground_truth, original_nan_indices, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizons_list, n_cv_splits, plot_params=None):
    results_by_horizon = []
    max_lags = max(n_lags_target, n_lags_exo)
    for horizon in tqdm(horizons_list, desc="  Horizons", leave=False):
        X = create_multivariate_lagged_features(df_city_data_imputed, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon)
        y = df_city_data_ground_truth[target_col].rename('target_h')
        df_processed = pd.concat([X, y], axis=1).dropna()
        if df_processed.empty or len(df_processed) < n_cv_splits * 2:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan}); continue
        y_proc = df_processed['target_h']; X_proc = df_processed.drop(columns=['target_h'])
        
        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        maes, rmses = [], []
        for _, (train_index, test_index) in enumerate(tscv.split(X_proc)):
            X_train, X_test = X_proc.iloc[train_index], X_proc.iloc[test_index]
            y_train, y_test = y_proc.iloc[train_index], y_proc.iloc[test_index]
            if y_test.empty: continue
            
            # --- LÓGICA DE FILTRADO PARA EVALUACIÓN ---
            # Máscara 1: El objetivo y_test debe ser un valor real (no un NaN original)
            target_is_real_mask = ~y_test.index.isin(original_nan_indices)
            
            # Máscara 2: La historia para predecir y_test DEBE contener valores imputados (si no es el baseline)
            if not original_nan_indices:
                history_is_dirty_mask = pd.Series(True, index=y_test.index) # Para el baseline, no aplicamos este filtro
            else:
                history_is_dirty_list = []
                for idx in y_test.index:
                    history_window = range(idx - horizon - max_lags, idx - horizon)
                    history_is_dirty_list.append(any(h_idx in original_nan_indices for h_idx in history_window))
                history_is_dirty_mask = pd.Series(history_is_dirty_list, index=y_test.index)

            # Máscara final: se deben cumplir ambas condiciones
            final_valid_mask = target_is_real_mask & history_is_dirty_mask
            y_test_valid = y_test[final_valid_mask]
            
            if y_test_valid.empty: continue
            
            model = LinearRegression().fit(X_train, y_train)
            predictions = model.predict(X_test)
            predictions_valid = predictions[final_valid_mask]
                
            maes.append(mean_absolute_error(y_test_valid, predictions_valid))
            rmses.append(np.sqrt(mean_squared_error(y_test_valid, predictions_valid)))
        
        results_by_horizon.append({'Horizon': horizon, 'MAE': np.mean(maes) if maes else np.nan, 'RMSE': np.mean(rmses) if rmses else np.nan})

        if plot_params and maes:
            final_model = LinearRegression().fit(X_proc, y_proc)
            all_features_full_df = create_multivariate_lagged_features(df_city_data_imputed, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon)
            generate_multi_step_forecasting_plots(df_imputed_data=df_city_data_imputed, df_ground_truth_data=df_city_data_ground_truth, model=final_model, X_all_features=all_features_full_df, original_nan_indices=original_nan_indices, city=plot_params['city'], scenario=plot_params['scenario'], method=plot_params['method'], rep_id=plot_params['rep_id'], horizon=horizon, output_dir=plot_params['output_dir'], n_examples=plot_params['n_examples'], history_len=plot_params['history_len'])
    return results_by_horizon

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month
    print(f"Original DataFrame '{ORIGINAL_DATA_FILE}' loaded.")
except FileNotFoundError:
    print(f"Error: '{ORIGINAL_DATA_FILE}' not found."); df_original_full = None

if df_original_full is not None:
    print("\n--- Forecasting with Original Data (Upper Bound Baseline) ---")
    method_name_baseline = 'Original_Data_Baseline'
    empty_nan_indices = set()
    for city_name in df_original_full['City'].unique():
        print(f"  City: {city_name}")
        df_city_ground_truth = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
            if col in df_city_ground_truth.columns and df_city_ground_truth[col].isnull().any():
                df_city_ground_truth[col] = df_city_ground_truth[col].ffill().bfill().fillna(df_original_full[col].mean())
        plot_parameters_baseline = {'city': city_name, 'scenario': 'N/A', 'method': method_name_baseline, 'rep_id': 'baseline', 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR, 'n_examples': N_PLOT_EXAMPLES, 'history_len': HISTORY_LEN_FOR_PLOT}
        horizon_metrics = train_predict_eval_multi_horizon(df_city_ground_truth, df_city_ground_truth, empty_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_parameters_baseline)
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({'Imputation_Method': method_name_baseline, 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE']})
        if horizon_metrics and not np.isnan(horizon_metrics[0]['MAE']): print(f"    Metrics (h={horizon_metrics[0]['Horizon']}): MAE={horizon_metrics[0]['MAE']:.4f}, RMSE={horizon_metrics[0]['RMSE']:.4f}")

print("\n--- Forecasting with Imputed Data ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    print(f"\n--- Processing Imputation Method: {method_name} ---")
    city_folders = [f for f in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, f))]
    for city_folder in tqdm(city_folders, desc=f"  Cities in {method_name}"):
        city_path = os.path.join(imputed_csvs_root, city_folder)
        original_city_name = city_folder.replace("_", " ")
        df_city_ground_truth = df_original_full[df_original_full['City'] == original_city_name].copy().reset_index(drop=True)
        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            repetition_files = glob.glob(os.path.join(scenario_path, "*.csv"))
            if not repetition_files: continue
            rep_file_path = repetition_files[0]
            file_name = os.path.basename(rep_file_path)
            rep_file_id_parts = [part for part in file_name.split('_') if 'rep' in part or 'seed' in part]
            rep_file_id = "_".join(rep_file_id_parts) if rep_file_id_parts else file_name
            nan_filename = f"{city_folder}_{scenario_folder}_{rep_file_id}_with_nans.csv" if rep_file_id_parts else None
            try:
                if nan_filename:
                    nan_file_path = os.path.join(SIMULATED_NANS_INPUT_DIR, city_folder, scenario_folder, nan_filename)
                    df_with_nans = pd.read_csv(nan_file_path)
                    original_nan_indices = set(df_with_nans[df_with_nans[TARGET_VARIABLE].isnull()].index)
                else: raise FileNotFoundError
            except FileNotFoundError:
                print(f"        WARNING: NaN file not found for {file_name}. Skipping impact evaluation checks."); original_nan_indices = set()
            try:
                df_rep = pd.read_csv(rep_file_path); df_rep['Date'] = pd.to_datetime(df_rep['Date'], infer_datetime_format=True, utc=True)
                df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
            except Exception as e: print(f"        Error loading {file_name}: {e}. Skipping."); continue
            df_city_rep = df_rep[df_rep['City'] == original_city_name].copy().reset_index(drop=True)
            if df_city_rep.isnull().values.any():
                for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
                    if col in df_city_rep.columns and df_city_rep[col].isnull().any(): df_city_rep[col] = df_city_rep[col].ffill().bfill().fillna(0)
            plot_parameters_rep = {'city': original_city_name, 'scenario': scenario_folder, 'method': method_name, 'rep_id': rep_file_id, 'output_dir': PREDICTION_PLOTS_OUTPUT_DIR, 'n_examples': N_PLOT_EXAMPLES, 'history_len': HISTORY_LEN_FOR_PLOT}
            horizon_metrics_rep = train_predict_eval_multi_horizon(df_city_rep, df_city_ground_truth, original_nan_indices, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS, plot_params=plot_parameters_rep)
            for hm_rep in horizon_metrics_rep:
                all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm_rep['Horizon'], 'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE']})

if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    detailed_output_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\nAll detailed prediction metrics saved to: {detailed_output_path}")
    df_summary = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Count=('MAE', 'count')).reset_index()
    for city_name in df_summary['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        df_city_summary = df_summary[df_summary['City'] == city_name]
        city_metrics_dir = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, city_name_safe)
        os.makedirs(city_metrics_dir, exist_ok=True)
        city_summary_path = os.path.join(city_metrics_dir, f'metrics_summary_{PREDICTION_METHOD_NAME}_{city_name_safe}.csv')
        df_city_summary.to_csv(city_summary_path, index=False)
        print(f"  Metrics summary for {city_name} saved to: {city_summary_path}")
    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"\nGlobal average metrics summary saved to: {global_summary_path}")
    print("--- Global Summary by Method and Horizon ---")
    print(df_global_summary.to_string())

script_end_time = time.time()
print(f"\nForecasting script completed. Total time: {(script_end_time - script_start_time)/60:.2f} minutes.")
print(f"Forecast plots saved in: {PREDICTION_PLOTS_OUTPUT_DIR}")

Original DataFrame 'c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' loaded.

--- Forecasting with Original Data (Upper Bound Baseline) ---
  City: Brasilia


    Metrics (h=12): MAE=7.7643, RMSE=9.8018
  City: Cairo


    Metrics (h=12): MAE=6.3841, RMSE=7.8923
  City: Dubai


    Metrics (h=12): MAE=8.3507, RMSE=10.7510
  City: London


    Metrics (h=12): MAE=5.1179, RMSE=6.5177
  City: New York


...

      Not enough valid windows (with imputed history and real future) to generate 1 plots for h=48.


  Cities in Random Forest:  50%|█████     | 3/6 [03:24<03:24, 68.14s/it]

      Not enough valid windows (with imputed history and real future) to generate 1 plots for h=60.


  Cities in Random Forest:  50%|█████     | 3/6 [04:09<04:09, 83.20s/it]


KeyboardInterrupt: 

Lineal con plots y csvs

In [2]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Configuración de Estilo para Gráficas ---
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (20, 8)
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['xtick.labelsize'] = 12
plt.rcParams['ytick.labelsize'] = 12
plt.rcParams['legend.fontsize'] = 12

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

N_LAGS_AQI = 60
N_LAGS_EXO = 60

PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
N_CV_SPLITS = 5

# --- Rutas (Asegúrate de que coincidan con tu estructura) ---
PROJECT_ROOT_DIR = os.getcwd()

ORIGINAL_DATA_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs')
ORIGINAL_DATA_FILE = os.path.join(ORIGINAL_DATA_DIR, 'Air_Quality.csv')

IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'regresionLinealMultipleLags_full'
PREDICTION_METHOD_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME)

# --- Rutas específicas para CSVs y Gráficos de Predicción ---
PREDICTION_METRICS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'metricas')
PREDICTION_CSVS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'predicciones_csv')
PREDICTION_PLOTS_OUTPUT_DIR = os.path.join(PREDICTION_METHOD_ROOT_DIR, 'plots')

os.makedirs(PREDICTION_METRICS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_CSVS_OUTPUT_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_OUTPUT_DIR, exist_ok=True)


# --- FUNCIÓN DE PLOTEO MODIFICADA ---
def plot_prediction_results(df_predictions, city, scenario, method, horizon, rep_id, output_dir):
    """Genera y guarda un gráfico comparando valores reales y predichos."""
    plt.figure()
    
    df_plot = df_predictions.sort_index()

    plt.plot(df_plot.index, df_plot['y_true'], label='Valores Reales (Ground Truth)', color='royalblue', alpha=0.9, zorder=2)
    plt.plot(df_plot.index, df_plot['y_pred'], label=f'Predicción (Horizonte {horizon}h)', color='darkorange', linestyle='--', alpha=0.8, zorder=3)
    
    plt.title(f'Predicción de {TARGET_VARIABLE} en {city} (Horizonte {horizon}h)\nImputación: {method} | Escenario: {scenario} | Rep: {rep_id}')
    plt.xlabel('Índice del Conjunto de Prueba')
    plt.ylabel(f'Valor de {TARGET_VARIABLE}')
    plt.legend()
    plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    
    # --- CAMBIO CLAVE: SANEAR EL NOMBRE DEL ESCENARIO PARA LA RUTA ---
    scenario_safe_for_path = scenario.replace('/', '_').replace('\\', '_')
    
    plot_save_dir = os.path.join(output_dir, method, city.replace(" ", "_"), scenario_safe_for_path)
    os.makedirs(plot_save_dir, exist_ok=True)
    
    plot_filename = f'pred_{city.replace(" ", "_")}_{scenario_safe_for_path}_{rep_id}_h{horizon}.png'
    plt.savefig(os.path.join(plot_save_dir, plot_filename), dpi=150)
    plt.close()

# --- Funciones Auxiliares (sin cambios) ---

def create_multivariate_lagged_features(df_full_city_data, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon):
    features_list = []
    for i in range(1, n_lags_target + 1):
        lagged_series = df_full_city_data[target_col].shift(i)
        lagged_series.name = f'{target_col}_lag{i}'
        features_list.append(lagged_series)
    for exo_col in exo_cols:
        if exo_col == target_col: continue
        for i in range(1, n_lags_exo + 1):
            lagged_series = df_full_city_data[exo_col].shift(i)
            lagged_series.name = f'{exo_col}_lag{i}'
            features_list.append(lagged_series)
    for tf_col in time_features_cols:
        if tf_col in df_full_city_data.columns:
            features_list.append(df_full_city_data[tf_col])
    df_features = pd.concat(features_list, axis=1)
    df_target = pd.DataFrame(df_full_city_data[target_col].shift(-horizon))
    df_target.columns = ['target_h']
    df_processed = pd.concat([df_features, df_target], axis=1)
    df_processed = df_processed.dropna()
    if df_processed.empty:
        return pd.DataFrame(), pd.Series(dtype='float64')
    y = df_processed['target_h']
    X = df_processed.drop(columns=['target_h'])
    return X, y

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_predict_eval_multi_horizon(df_city_data_input, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizons_list, n_cv_splits):
    results_by_horizon = []
    predictions_by_horizon = {}
    max_lags = max(n_lags_target, n_lags_exo)
    min_len = max_lags + max(horizons_list) + n_cv_splits * 10
    
    if len(df_city_data_input) < min_len:
        print(f"    Datos insuficientes. Mínimo requerido: {min_len}, disponible: {len(df_city_data_input)}")
        for h in horizons_list:
            results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon, predictions_by_horizon

    for horizon in horizons_list:
        X, y = create_multivariate_lagged_features(df_city_data_input, target_col, exo_cols, time_features_cols, n_lags_target, n_lags_exo, horizon)

        if X.empty or y.empty or len(X) < n_cv_splits * 2:
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        tscv = TimeSeriesSplit(n_splits=n_cv_splits)
        horizon_maes, horizon_rmses, horizon_mases = [], [], []
        
        all_fold_predictions_df = [] 

        for fold, (train_index, test_index) in enumerate(tscv.split(X)):
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]

            if X_train.empty or y_train.empty or X_test.empty or y_test.empty or len(y_train) <=1:
                continue

            model = LinearRegression()
            try:
                model.fit(X_train, y_train)
                predictions = model.predict(X_test)
                
                fold_preds_df = pd.DataFrame({'y_true': y_test, 'y_pred': predictions}, index=y_test.index)
                all_fold_predictions_df.append(fold_preds_df)

                original_series_train_end_idx = X_train.index[-1]
                y_train_for_naive = df_city_data_input[target_col].iloc[:original_series_train_end_idx + 1].values
                
                mae = mean_absolute_error(y_test, predictions)
                rmse = np.sqrt(mean_squared_error(y_test, predictions))
                mase = calculate_mase(y_test.values, predictions, y_train_for_naive)

                horizon_maes.append(mae)
                horizon_rmses.append(rmse)
                if not np.isnan(mase): horizon_mases.append(mase)
            except Exception as e:
                print(f"        Error en Fold {fold+1} para horizonte {horizon}h: {e}")
                continue
        
        if all_fold_predictions_df:
            predictions_by_horizon[horizon] = pd.concat(all_fold_predictions_df)
            
        results_by_horizon.append({
            'Horizon': horizon,
            'MAE': np.nanmean(horizon_maes) if horizon_maes else np.nan,
            'RMSE': np.nanmean(horizon_rmses) if horizon_rmses else np.nan,
            'MASE': np.nanmean(horizon_mases) if horizon_mases else np.nan
        })
    return results_by_horizon, predictions_by_horizon

# --- Script Principal (sin cambios) ---
all_prediction_metrics_detailed = []
script_start_time = time.time()

try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'])
    df_original_full['Hour'] = df_original_full['Date'].dt.hour
    df_original_full['DayOfWeek'] = df_original_full['Date'].dt.dayofweek
    df_original_full['Month'] = df_original_full['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'.")
    df_original_full = None

if df_original_full is not None:
    print("\n--- Predicción Multivariante con Datos Originales (Baseline) ---")
    method_name_baseline = 'Original_FFill_Mean'
    for city_name in df_original_full['City'].unique():
        print(f"  Ciudad: {city_name}")
        df_city_original = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        
        cols_to_fill = [TARGET_VARIABLE] + EXOGENOUS_COLS
        for col in cols_to_fill:
            if col in df_city_original.columns and df_city_original[col].isnull().any():
                df_city_original[col] = df_city_original[col].ffill().bfill()
                if df_city_original[col].isnull().any():
                    global_mean = df_original_full[col].mean()
                    df_city_original[col] = df_city_original[col].fillna(global_mean)

        if df_city_original[cols_to_fill].isnull().values.any():
            print(f"    ADVERTENCIA: No se pudieron rellenar todos los NaNs en el baseline para {city_name}. Omitiendo ciudad.")
            continue
        
        horizon_metrics, predictions = train_predict_eval_multi_horizon(
            df_city_original, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
            N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS
        )
        
        city_name_safe = city_name.replace(" ", "_")
        for h, df_preds in predictions.items():
            if not df_preds.empty:
                # --- CAMBIO CLAVE: SANEAR EL NOMBRE DEL ESCENARIO PARA LA RUTA DEL CSV ---
                scenario_safe_for_csv_path = "N_A".replace('/', '_').replace('\\', '_')
                pred_csv_dir = os.path.join(PREDICTION_CSVS_OUTPUT_DIR, method_name_baseline, city_name_safe, scenario_safe_for_csv_path)
                os.makedirs(pred_csv_dir, exist_ok=True)
                pred_csv_path = os.path.join(pred_csv_dir, f'pred_{city_name_safe}_baseline_h{h}.csv')
                df_preds.to_csv(pred_csv_path)
                
                if h == PREDICTION_HORIZONS[-1]:
                     plot_prediction_results(df_preds, city_name, "N/A", method_name_baseline, h, "baseline", PREDICTION_PLOTS_OUTPUT_DIR)

        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({
                'Imputation_Method': method_name_baseline, 'City': city_name, 'Scenario': 'N/A',
                'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'],
                'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']
            })
        if horizon_metrics:
            if not np.isnan(horizon_metrics[0]['MAE']):
                print(f"    Métricas (Baseline, {horizon_metrics[0]['Horizon']}h): MAE={horizon_metrics[0]['MAE']:.4f}, RMSE={horizon_metrics[0]['RMSE']:.4f}, MASE={horizon_metrics[0]['MASE']:.4f}")
            if len(horizon_metrics) > 1 and not np.isnan(horizon_metrics[-1]['MAE']):
                print(f"    Métricas (Baseline, {horizon_metrics[-1]['Horizon']}h): MAE={horizon_metrics[-1]['RMSE']:.4f}, MASE={horizon_metrics[-1]['MASE']:.4f}")

imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
print(f"\n--- Predicción Multivariante con Datos Imputados ({len(imputation_methods)} métodos) ---")
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Procesando método de imputación: {method_name} ---")
    
    for city_folder in os.listdir(imputed_csvs_root):
        city_path = os.path.join(imputed_csvs_root, city_folder)
        if not os.path.isdir(city_path): continue
        original_city_name = city_folder.replace("_", " ")
        print(f"  Ciudad: {original_city_name}")

        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")

            repetition_files = glob.glob(os.path.join(scenario_path, "*.csv"))
            if not repetition_files: continue
            
            scenario_metrics_accumulator = {h: {'maes': [], 'rmses': [], 'mases': []} for h in PREDICTION_HORIZONS}

            for i, rep_file_path in enumerate(repetition_files):
                file_name = os.path.basename(rep_file_path)
                rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                
                try:
                    df_rep = pd.read_csv(rep_file_path)
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                    df_rep['Hour'] = df_rep['Date'].dt.hour
                    df_rep['DayOfWeek'] = df_rep['Date'].dt.dayofweek
                    df_rep['Month'] = df_rep['Date'].dt.month
                except Exception as e:
                    print(f"        Error cargando o procesando {file_name}: {e}. Omitiendo."); continue
                
                df_city_rep = df_rep[df_rep['City'] == original_city_name].copy().reset_index(drop=True)

                if df_city_rep.isnull().values.any():
                    print(f"        ADVERTENCIA: NaNs encontrados en {file_name} DESPUÉS de imputar. Rellenando con ffill/bfill.")
                    for col in [TARGET_VARIABLE] + EXOGENOUS_COLS:
                        if col in df_city_rep.columns and df_city_rep[col].isnull().any():
                             df_city_rep[col] = df_city_rep[col].ffill().bfill()
                    if df_city_rep.isnull().values.any():
                        print(f"        ERROR: NaNs persisten en {file_name}. Omitiendo archivo."); continue
                
                horizon_metrics_rep, predictions_rep = train_predict_eval_multi_horizon(
                    df_city_rep, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES,
                    N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS
                )
                
                city_name_safe_loop = original_city_name.replace(" ", "_")
                for h, df_preds_rep in predictions_rep.items():
                    if not df_preds_rep.empty:
                        # --- CAMBIO CLAVE: SANEAR EL NOMBRE DEL ESCENARIO PARA LA RUTA DEL CSV ---
                        scenario_safe_for_csv_path = scenario_folder.replace('/', '_').replace('\\', '_')
                        pred_csv_dir = os.path.join(PREDICTION_CSVS_OUTPUT_DIR, method_name, city_name_safe_loop, scenario_safe_for_csv_path)
                        os.makedirs(pred_csv_dir, exist_ok=True)
                        pred_csv_filename = f'pred_{city_name_safe_loop}_{scenario_safe_for_csv_path}_{rep_file_id}_h{h}.csv'
                        df_preds_rep.to_csv(os.path.join(pred_csv_dir, pred_csv_filename))
                        
                        is_first_repetition = ("_rep1_" in file_name) or (i == 0)
                        if is_first_repetition and h == PREDICTION_HORIZONS[-1]:
                            plot_prediction_results(df_preds_rep, original_city_name, scenario_folder, method_name, h, rep_file_id, PREDICTION_PLOTS_OUTPUT_DIR)

                for hm_rep in horizon_metrics_rep:
                    h = hm_rep['Horizon']
                    all_prediction_metrics_detailed.append({
                        'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder,
                        'Repetition_File_ID': rep_file_id, 'Horizon_h': h,
                        'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']
                    })
                    if not np.isnan(hm_rep.get('MAE')): scenario_metrics_accumulator[h]['maes'].append(hm_rep['MAE'])
                    if not np.isnan(hm_rep.get('RMSE')): scenario_metrics_accumulator[h]['rmses'].append(hm_rep['RMSE'])
                    if not np.isnan(hm_rep.get('MASE')): scenario_metrics_accumulator[h]['mases'].append(hm_rep['MASE'])

            print(f"      Promedios para Escenario '{scenario_folder}':")
            for h_summary in [PREDICTION_HORIZONS[0], PREDICTION_HORIZONS[-1]]:
                if h_summary in scenario_metrics_accumulator:
                    avg_mae = np.nanmean(scenario_metrics_accumulator[h_summary]['maes']) if scenario_metrics_accumulator[h_summary]['maes'] else np.nan
                    if not np.isnan(avg_mae):
                        avg_rmse = np.nanmean(scenario_metrics_accumulator[h_summary]['rmses'])
                        avg_mase = np.nanmean(scenario_metrics_accumulator[h_summary]['mases'])
                        print(f"        Horizonte {h_summary}h: MAE={avg_mae:.4f}, RMSE={avg_rmse:.4f}, MASE={avg_mase:.4f}")

if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)

    detailed_output_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\nTodas las métricas de predicción detalladas guardadas en: {detailed_output_path}")

    df_summary = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'), Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'), Count=('MAE', 'count')).reset_index()

    for city_name in df_summary['City'].unique():
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        df_city_summary = df_summary[df_summary['City'] == city_name]
        city_metrics_dir = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, city_name_safe)
        os.makedirs(city_metrics_dir, exist_ok=True)
        city_summary_path = os.path.join(city_metrics_dir, f'metrics_summary_{PREDICTION_METHOD_NAME}_{city_name_safe}.csv')
        df_city_summary.to_csv(city_summary_path, index=False)
        print(f"  Resumen de métricas para {city_name} guardado en: {city_summary_path}")

    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"\nResumen global de métricas promedio guardado en: {global_summary_path}")
    print("--- Resumen Global por Método y Horizonte ---")
    print(df_global_summary.to_string())
    
    df_metrics_for_avg = df_all_metrics[df_all_metrics['Scenario'] != 'N/A'].copy()
    df_avg_by_method = df_metrics_for_avg.groupby(['Imputation_Method', 'Horizon_h']).agg(Avg_Across_Scenarios_MAE=('MAE', 'mean'), Avg_Across_Scenarios_RMSE=('RMSE', 'mean'), Avg_Across_Scenarios_MASE=('MASE', 'mean')).reset_index()

    df_baseline = df_all_metrics[df_all_metrics['Imputation_Method'] == 'Original_FFill_Mean'].copy()
    if not df_baseline.empty:
        df_baseline_summary = df_baseline.groupby(['Imputation_Method', 'Horizon_h']).agg(Avg_Across_Scenarios_MAE=('MAE', 'mean'), Avg_Across_Scenarios_RMSE=('RMSE', 'mean'), Avg_Across_Scenarios_MASE=('MASE', 'mean')).reset_index()
        df_avg_by_method = pd.concat([df_avg_by_method, df_baseline_summary], ignore_index=True).sort_values(by=['Imputation_Method', 'Horizon_h'])

    avg_by_method_path = os.path.join(PREDICTION_METRICS_OUTPUT_DIR, f'ALL_prediction_metrics_avg_by_method_{PREDICTION_METHOD_NAME}.csv')
    df_avg_by_method.to_csv(avg_by_method_path, index=False)
    print(f"\nResumen de métricas promediadas por método guardado en: {avg_by_method_path}")
    print("--- Resumen por Método (promedio de ciudades, escenarios y repeticiones) ---")
    print(df_avg_by_method.to_string())

script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")
print(f"CSVs de predicción guardados en: {PREDICTION_CSVS_OUTPUT_DIR}")
print(f"Gráficos de predicción guardados en: {PREDICTION_PLOTS_OUTPUT_DIR}")

DataFrame original 'c:\Users\fdemiguel\Documentos\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado.

--- Predicción Multivariante con Datos Originales (Baseline) ---
  Ciudad: Brasilia
    Métricas (Baseline, 12h): MAE=5.7678, RMSE=7.3499, MASE=3.4510
    Métricas (Baseline, 60h): MAE=12.8170, MASE=6.1020
  Ciudad: Cairo
    Métricas (Baseline, 12h): MAE=6.4057, RMSE=7.9000, MASE=6.7811
    Métricas (Baseline, 60h): MAE=17.0310, MASE=14.5781
  Ciudad: Dubai
    Métricas (Baseline, 12h): MAE=6.9764, RMSE=9.0235, MASE=8.0043
    Métricas (Baseline, 60h): MAE=27.0626, MASE=25.3508
  Ciudad: London
    Métricas (Baseline, 12h): MAE=4.9318, RMSE=6.2782, MASE=4.1129
    Métricas (Baseline, 60h): MAE=10.5137, MASE=6.2679
  Ciudad: New York
    Métricas (Baseline, 12h): MAE=7.2511, RMSE=9.0380, MASE=4.5129
    Métricas (Baseline, 60h): MAE=16.5922, MASE=8.2775
  Ciudad: Sydney
    Métricas (Baseline, 12h): MAE=5.9507, RMSE=7.9373, MASE=4.9745
    Métricas (Baseline, 60h): MAE=11.7803, M

SARIMAX

In [1]:
import pandas as pd
import numpy as np
import os
import json
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning) # Para statsmodels

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

PREDICTION_HORIZONS = [12, 24, 36, 48, 60]

# --- MODIFICACIONES PARA PRUEBAS RÁPIDAS ---
TARGET_CITY_FILTER = "Brasilia" # None para procesar todas las ciudades
N_REPETITIONS_TO_PROCESS = 1
N_CV_SPLITS = 2

# NUEVO: Lista de escenarios específicos a procesar. Si es None, se procesan todos.
SCENARIOS_TO_PROCESS = None # ['BlockRows_12h_20percent'] 
# Ejemplo para procesar dos: SCENARIOS_TO_PROCESS = ['BlockRows_12h_20percent', 'RandomRows_20percent']
# Ejemplo para procesar todos: SCENARIOS_TO_PROCESS = None
# -------------------------------------------

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
ORDERS_JSON_INPUT_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', 'SARIMA', 'model_orders')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'SARIMAX_pred_test'
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)


# --- Funciones Auxiliares ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_predict_eval_sarimax_multi_horizon(
    df_city_data_input, target_col, exo_cols, time_features_cols,
    sarima_order, sarima_seasonal_order, horizons_list, n_cv_splits
):
    results_by_horizon = []
    y = df_city_data_input[target_col]
    X = df_city_data_input[exo_cols + time_features_cols]
    
    min_len = (sum(sarima_order) + sum(sarima_seasonal_order)) * 2 + max(horizons_list) + n_cv_splits
    if len(y) < min_len:
        print(f"    Datos insuficientes. Mínimo: ~{min_len}, disponible: {len(y)}")
        for h in horizons_list: results_by_horizon.append({'Horizon': h, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
        return results_by_horizon
        
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons_list))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons_list}

    print(f"      -> Iniciando validación cruzada con {n_cv_splits} fold(s)...")
    for fold, (train_index, test_index) in enumerate(tscv.split(X)):
        print(f"        -> Fold {fold + 1}/{n_cv_splits}...")
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]

        if y_train.empty or X_train.empty:
            print("        -> Fold omitido por datos vacíos.")
            continue
        try:
            print("          -> Ajustando modelo SARIMAX...")
            model = SARIMAX(y_train, exog=X_train, order=sarima_order, seasonal_order=sarima_seasonal_order,
                            enforce_stationarity=False, enforce_invertibility=False)
            results_fit = model.fit(disp=False, maxiter=200)
            print("          -> Realizando predicciones...")
            forecasts = results_fit.get_forecast(steps=max(horizons_list), exog=X_test.head(max(horizons_list)))
            forecast_values = forecasts.predicted_mean

            for h in horizons_list:
                y_true_h, y_pred_h = y_test.head(h).values, forecast_values.head(h).values
                if len(y_true_h) < h: continue
                mae = mean_absolute_error(y_true_h, y_pred_h)
                rmse = np.sqrt(mean_squared_error(y_true_h, y_pred_h))
                mase = calculate_mase(y_true_h, y_pred_h, y_train.values)
                accumulated_metrics[h]['maes'].append(mae)
                accumulated_metrics[h]['rmses'].append(rmse)
                if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        except Exception as e:
            print(f"        -> Error en Fold {fold+1}: {e}")
            continue
    
    for h in horizons_list:
        results_by_horizon.append({'Horizon': h, 'MAE': np.nanmean(accumulated_metrics[h]['maes']) if accumulated_metrics[h]['maes'] else np.nan, 'RMSE': np.nanmean(accumulated_metrics[h]['rmses']) if accumulated_metrics[h]['rmses'] else np.nan, 'MASE': np.nanmean(accumulated_metrics[h]['mases']) if accumulated_metrics[h]['mases'] else np.nan})
    return results_by_horizon

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()

# 1. Cargar y preprocesar DataFrame Original (Baseline)
try:
    df_original_full = pd.read_csv(ORIGINAL_DATA_FILE)
    df_original_full['Date'] = pd.to_datetime(df_original_full['Date'], utc=True)
    df_original_full['Hour'], df_original_full['DayOfWeek'], df_original_full['Month'] = df_original_full['Date'].dt.hour, df_original_full['Date'].dt.dayofweek, df_original_full['Date'].dt.month
    print(f"DataFrame original '{ORIGINAL_DATA_FILE}' cargado.")
except FileNotFoundError:
    print(f"Error: No se encontró '{ORIGINAL_DATA_FILE}'.")
    df_original_full = None

if TARGET_CITY_FILTER:
    cities_to_process = [TARGET_CITY_FILTER]
    print(f"\nATENCIÓN: Se procesará únicamente la ciudad: {TARGET_CITY_FILTER}")
else:
    cities_to_process = df_original_full['City'].unique() if df_original_full is not None else []
    print("\nATENCIÓN: Se procesarán todas las ciudades encontradas.")

if df_original_full is not None:
    print("\n--- Predicción SARIMAX con Datos Originales (Baseline) ---")
    for city_name in cities_to_process:
        if city_name not in df_original_full['City'].unique(): continue
        print(f"  Procesando ciudad (Baseline): {city_name}")
        df_city_original = df_original_full[df_original_full['City'] == city_name].copy().reset_index(drop=True)
        cols_to_fill = [TARGET_VARIABLE] + EXOGENOUS_COLS
        for col in cols_to_fill:
            if col in df_city_original.columns and df_city_original[col].isnull().any():
                df_city_original[col] = df_city_original[col].interpolate(method='linear').ffill().bfill()
        if df_city_original[cols_to_fill].isnull().values.any():
            print(f"    ADVERTENCIA: No se pudieron rellenar todos los NaNs para {city_name}. Omitiendo."); continue
        
        city_name_safe = city_name.replace(" ", "_").replace("-", "_")
        order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
        try:
            with open(order_file_path, 'r') as f: orders = json.load(f)
            order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
            print(f"    -> Órdenes cargadas para {city_name}: {order_tuple}{seasonal_order_tuple}")
        except Exception:
            print(f"    -> Órdenes no encontradas. Usando (1,1,1)(1,1,0,24)."); order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)
        
        horizon_metrics = train_predict_eval_sarimax_multi_horizon(df_city_original, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS)
        for hm in horizon_metrics:
            all_prediction_metrics_detailed.append({'Imputation_Method': 'Original_Interpolate', 'City': city_name, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']})
        print(f"    -> Finalizado Baseline para {city_name}.")

# 2. Iterar sobre todos los métodos de imputación y sus archivos
print("\n--- Predicción SARIMAX con Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]

for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    print(f"\n--- Procesando método de imputación: {method_name} ---")
    
    city_folders_to_process = [c for c in os.listdir(imputed_csvs_root) if (TARGET_CITY_FILTER is None or c == TARGET_CITY_FILTER.replace(" ", "_"))]

    for city_folder in city_folders_to_process:
        city_path, original_city_name = os.path.join(imputed_csvs_root, city_folder), city_folder.replace("_", " ")
        if not os.path.isdir(city_path): continue
        print(f"  Procesando ciudad: {original_city_name}")
        
        city_name_safe = original_city_name.replace(" ", "_").replace("-", "_")
        order_file_path = os.path.join(ORDERS_JSON_INPUT_DIR, f'orders_{city_name_safe}_{TARGET_VARIABLE}.json')
        try:
            with open(order_file_path, 'r') as f: orders = json.load(f)
            order_tuple, seasonal_order_tuple = tuple(orders['order']), tuple(orders['seasonal_order'])
            print(f"    -> Órdenes cargadas para {original_city_name}: {order_tuple}{seasonal_order_tuple}")
        except Exception:
            print(f"    -> Órdenes no encontradas. Usando (1,1,1)(1,1,0,24)."); order_tuple, seasonal_order_tuple = (1,1,1), (1,1,0,24)

        # SOLUCIÓN: Filtrar los escenarios a procesar
        all_scenarios = [s for s in os.listdir(city_path) if os.path.isdir(os.path.join(city_path, s))]
        scenarios_to_run = all_scenarios if SCENARIOS_TO_PROCESS is None else [s for s in all_scenarios if any(filt in s for filt in SCENARIOS_TO_PROCESS)]
        print(f"    -> Escenarios a procesar: {scenarios_to_run if scenarios_to_run else 'Ninguno'}")

        for scenario_folder in scenarios_to_run:
            scenario_path = os.path.join(city_path, scenario_folder)
            print(f"    Procesando escenario: {scenario_folder}")
            
            all_files = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))
            files_to_process = all_files[:N_REPETITIONS_TO_PROCESS]
            if not files_to_process: print(f"      -> No se encontraron archivos para la repetición 1."); continue

            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                print(f"      -> Procesando archivo: {file_name}")
                rep_file_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                try:
                    df_rep = pd.read_csv(rep_file_path)
                    df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                    df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
                except Exception as e: print(f"        -> Error cargando {file_name}: {e}."); continue
                
                if df_rep.isnull().values.any():
                    print(f"        -> ADVERTENCIA: NaNs en {file_name}. Rellenando con ffill/bfill.")
                    df_rep = df_rep.ffill().bfill()
                    if df_rep.isnull().values.any(): continue
                
                horizon_metrics_rep = train_predict_eval_sarimax_multi_horizon(df_rep, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, order_tuple, seasonal_order_tuple, PREDICTION_HORIZONS, N_CV_SPLITS)
                for hm_rep in horizon_metrics_rep:
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': original_city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_file_id, 'Horizon_h': hm_rep['Horizon'], 'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']})

# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_output_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_output_path, index=False)
    print(f"\n[OK] Todas las métricas detalladas guardadas en: {detailed_output_path}")

    df_summary_by_city = df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean'), Std_MAE=('MAE', 'std'), Std_RMSE=('RMSE', 'std'), Std_MASE=('MASE', 'std'), Count=('MAE', 'count')).reset_index()
    summary_by_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_summary_by_city.to_csv(summary_by_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_by_city_path}")

    df_global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    df_global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")
    print("\n--- Resumen Global por Método y Horizonte ---\n", df_global_summary)

    df_avg_over_horizons = df_global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    avg_over_horizons_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    df_avg_over_horizons.to_csv(avg_over_horizons_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {avg_over_horizons_path}")
    print("\n--- Resumen Final por Método (promedio de todas las ciudades, escenarios y horizontes) ---\n", df_avg_over_horizons)

script_end_time = time.time()
print(f"\nScript de predicción completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

DataFrame original 'c:\Users\felix\Documents\MÁSTER\TFM\TFM_Ambiental\AIR QUALITY\CSVs\Air_Quality.csv' cargado.

ATENCIÓN: Se procesará únicamente la ciudad: Brasilia

--- Predicción SARIMAX con Datos Originales (Baseline) ---
  Procesando ciudad (Baseline): Brasilia
    -> Órdenes cargadas para Brasilia: (1, 0, 0)(1, 0, 1, 24)
      -> Iniciando validación cruzada con 2 fold(s)...
        -> Fold 1/2...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
        -> Fold 2/2...
          -> Ajustando modelo SARIMAX...
          -> Realizando predicciones...
    -> Finalizado Baseline para Brasilia.

--- Predicción SARIMAX con Datos Imputados ---

--- Procesando método de imputación: interpolacionLineal ---
  Procesando ciudad: Brasilia
    -> Órdenes cargadas para Brasilia: (1, 0, 0)(1, 0, 1, 24)
    -> Escenarios a procesar: ['BlockRows_12h_20percent', 'BlockRows_24h_20percent', 'BlockRows_36h_20percent', 'BlockRows_48h_20percent', 'BlockRows_60h_20percen

KeyboardInterrupt: 

Random Forest

In [10]:
import pandas as pd
import numpy as np
import os
import random
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
import time
import glob
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
EXOGENOUS_COLS = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10']
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']

N_LAGS_AQI = 60
N_LAGS_EXO = 60

PREDICTION_HORIZONS = [12, 24, 36, 48, 60]

# --- MODIFICACIONES PARA PRUEBAS RÁPIDAS ---
TARGET_CITY_FILTER = "Brasilia"
N_REPETITIONS_TO_PROCESS = 5
N_CV_SPLITS = 5                 
SCENARIOS_TO_PROCESS = None # ['BlockRows_12h_20percent']
# -------------------------------------------

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'RandomForest_pred'
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)

# --- Funciones Auxiliares ---
# SOLUCIÓN: Reescribir la función para evitar la fragmentación del DataFrame
def create_multivariate_lagged_features_rf(df, target_col, exo_cols, time_cols, n_lags_target, n_lags_exo):
    """
    Crea un DataFrame con todas las features y lags de forma eficiente usando pd.concat.
    """
    feature_series_list = []
    
    # Features de tiempo
    for tf_col in time_cols:
        if tf_col in df.columns:
            feature_series_list.append(df[tf_col])
    
    # Lags del target
    for i in range(1, n_lags_target + 1):
        series = df[target_col].shift(i)
        series.name = f'{target_col}_lag{i}'
        feature_series_list.append(series)
        
    # Lags y contemporáneos de exógenas
    for exo_col in exo_cols:
        if exo_col != target_col:
            # Valor contemporáneo
            series_t0 = df[exo_col]
            series_t0.name = f'{exo_col}_t0'
            feature_series_list.append(series_t0)
            # Lags
            for i in range(1, n_lags_exo + 1):
                series_lag = df[exo_col].shift(i)
                series_lag.name = f'{exo_col}_lag{i}'
                feature_series_list.append(series_lag)
    
    # Unir todas las series de features de una sola vez
    return pd.concat(feature_series_list, axis=1)


def calculate_mase(y_true, y_pred, y_train_for_naive):
    y_train_no_nan = y_train_for_naive.dropna()
    if len(y_train_no_nan) <= 1: return np.nan
    mae_naive = np.mean(np.abs(np.diff(y_train_no_nan.values)))
    if mae_naive < 1e-9: return np.nan
    return np.mean(np.abs(y_true - y_pred)) / mae_naive

def train_predict_eval_rf_multi_horizon(df_city_data, target_col, exo_cols, time_features, n_lags_target, n_lags_exo, horizons, n_cv_splits):
    results_by_horizon = []
    
    print(f"      -> Creando features con lags...")
    features = create_multivariate_lagged_features_rf(df_city_data, target_col, exo_cols, time_features, n_lags_target, n_lags_exo)
    
    for horizon in horizons:
        target = df_city_data[target_col].shift(-horizon)
        
        full_dataset = pd.concat([features, target], axis=1)
        full_dataset.columns = list(features.columns) + ['target_h']
        
        full_dataset_clean = full_dataset.dropna()

        X = full_dataset_clean.drop(columns=['target_h'])
        y = full_dataset_clean['target_h']
        
        print(f"      -> H={horizon}h: Creadas {len(X)} muestras limpias para entrenar/evaluar.")
        if X.empty or y.empty or len(X) < n_cv_splits * 10:
            print(f"      -> H={horizon}h: Omitido por falta de datos post-procesamiento.")
            results_by_horizon.append({'Horizon': horizon, 'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan})
            continue

        tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons))
        h_metrics = {'maes': [], 'rmses': [], 'mases': []}
        
        for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
            X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
            X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

            X_test_h = X_test.head(horizon)
            y_test_h = y_test.head(horizon)

            if X_train.empty or y_train.empty or X_test_h.empty or y_test_h.empty or len(y_test_h) < horizon:
                continue

            model = RandomForestRegressor(n_estimators=150, max_depth=15, min_samples_split=5, min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=-1)
            try:
                model.fit(X_train, y_train)
                preds = model.predict(X_test_h)
                
                h_metrics['maes'].append(mean_absolute_error(y_test_h, preds))
                h_metrics['rmses'].append(np.sqrt(mean_squared_error(y_test_h, preds)))
                
                original_y_train_indices = y.iloc[train_idx].index
                mase = calculate_mase(y_test_h.values, preds, df_city_data.loc[original_y_train_indices, target_col])
                if not np.isnan(mase): h_metrics['mases'].append(mase)
            except Exception as e:
                print(f"        -> Error en Fold {fold+1} (H={horizon}): {e}")
        
        results_by_horizon.append({'Horizon': horizon, 'MAE': np.nanmean(h_metrics['maes']) if h_metrics['maes'] else np.nan, 'RMSE': np.nanmean(h_metrics['rmses']) if h_metrics['rmses'] else np.nan, 'MASE': np.nanmean(h_metrics['mases']) if h_metrics['mases'] else np.nan})
    return results_by_horizon

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()

datasets_to_process = []
datasets_to_process.append({'method': 'Baseline_Cleaned', 'path': ORIGINAL_DATA_FILE, 'type': 'baseline'})
datasets_to_process.append({'method': 'RF_Handles_NaNs', 'path': SIMULATED_DATA_ROOT_DIR, 'type': 'simulated'})
for method_name in [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]:
    datasets_to_process.append({'method': method_name, 'path': os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs'), 'type': 'imputed'})

for dataset_info in datasets_to_process:
    method_name, base_path, data_type = dataset_info['method'], dataset_info['path'], dataset_info['type']
    print(f"\n--- Procesando datos de: {method_name} ---")

    if not os.path.exists(base_path):
        print(f"  Ruta no encontrada: {base_path}. Omitiendo."); continue

    if data_type == 'baseline':
        df = pd.read_csv(base_path)
        df['Date'] = pd.to_datetime(df['Date'], utc=True)
        cities = [TARGET_CITY_FILTER] if TARGET_CITY_FILTER else df['City'].unique()
        for city in cities:
            print(f"  Procesando ciudad (Baseline): {city}")
            df_city = df[df['City'] == city].copy()
            df_city = df_city.interpolate(method='linear').ffill().bfill()
            if df_city[[TARGET_VARIABLE] + EXOGENOUS_COLS].isnull().values.any(): continue
            df_city['Hour'], df_city['DayOfWeek'], df_city['Month'] = df_city['Date'].dt.hour, df_city['Date'].dt.dayofweek, df_city['Date'].dt.month
            horizon_metrics = train_predict_eval_rf_multi_horizon(df_city, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS)
            for hm in horizon_metrics:
                all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': hm['Horizon'], 'MAE': hm['MAE'], 'RMSE': hm['RMSE'], 'MASE': hm['MASE']})
    else:
        city_folders = [c for c in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, c)) and (TARGET_CITY_FILTER is None or c == TARGET_CITY_FILTER.replace(" ", "_"))]
        for city_folder in city_folders:
            city_path, city_name = os.path.join(base_path, city_folder), city_folder.replace("_", " ")
            print(f"  Procesando ciudad: {city_name}")
            all_scenarios = [s for s in os.listdir(city_path) if os.path.isdir(os.path.join(city_path, s))]
            scenarios_to_run = all_scenarios if SCENARIOS_TO_PROCESS is None else [s for s in all_scenarios if any(filt in s for filt in SCENARIOS_TO_PROCESS)]
            
            for scenario_folder in scenarios_to_run:
                scenario_path = os.path.join(city_path, scenario_folder)
                print(f"    Procesando escenario: {scenario_folder}")
                files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
                if not files_to_process: continue

                for rep_file_path in files_to_process:
                    file_name = os.path.basename(rep_file_path)
                    print(f"      -> Procesando archivo: {file_name}")
                    try:
                        df_rep = pd.read_csv(rep_file_path)
                        df_rep['Date'] = pd.to_datetime(df_rep['Date'], utc=True)
                        df_rep['Hour'], df_rep['DayOfWeek'], df_rep['Month'] = df_rep['Date'].dt.hour, df_rep['Date'].dt.dayofweek, df_rep['Date'].dt.month
                    except Exception as e: print(f"        -> Error cargando {file_name}: {e}."); continue
                    
                    if data_type == 'imputed' and df_rep.isnull().values.any():
                        print(f"        -> ADVERTENCIA: NaNs en archivo imputado {file_name}. Rellenando."); df_rep = df_rep.ffill().bfill()
                    
                    if df_rep.isnull().values.any() and data_type != 'simulated':
                        print(f"        -> ERROR: NaNs persisten en archivo imputado {file_name}. Omitiendo."); continue

                    horizon_metrics_rep = train_predict_eval_rf_multi_horizon(df_rep, TARGET_VARIABLE, EXOGENOUS_COLS, TIME_FEATURES, N_LAGS_AQI, N_LAGS_EXO, PREDICTION_HORIZONS, N_CV_SPLITS)
                    for hm_rep in horizon_metrics_rep:
                        all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': file_name, 'Horizon_h': hm_rep['Horizon'], 'MAE': hm_rep['MAE'], 'RMSE': hm_rep['RMSE'], 'MASE': hm_rep['MASE']})

# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_path, index=False)
    print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")

    summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")

    global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")
    print("\n--- Resumen Global por Método y Horizonte ---\n", global_summary)

    final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    final_summary.to_csv(final_summary_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
    print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary)

script_end_time = time.time()
print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")


--- Procesando datos de: Baseline_Cleaned ---
  Procesando ciudad (Baseline): Brasilia
      -> Creando features con lags...
      -> H=12h: Creadas 8712 muestras limpias para entrenar/evaluar.
      -> H=24h: Creadas 8700 muestras limpias para entrenar/evaluar.
      -> H=36h: Creadas 8688 muestras limpias para entrenar/evaluar.
      -> H=48h: Creadas 8676 muestras limpias para entrenar/evaluar.
      -> H=60h: Creadas 8664 muestras limpias para entrenar/evaluar.

--- Procesando datos de: RF_Handles_NaNs ---
  Procesando ciudad: Brasilia
    Procesando escenario: BlockRows_12h_20percent
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep1_seed2510790048_with_nans.csv
      -> Creando features con lags...
      -> H=12h: Creadas 1559 muestras limpias para entrenar/evaluar.
      -> H=24h: Creadas 1590 muestras limpias para entrenar/evaluar.
      -> H=36h: Creadas 1630 muestras limpias para entrenar/evaluar.
      -> H=48h: Creadas 1620 muestras limpias para entrenar/ev

LSTM-VAE

In [ ]:
import pandas as pd
import numpy as np
import os
import json
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import tensorflow as tf
import keras # Keras 3 de alto nivel
from keras import layers, Model
from keras import backend as K
import time
import glob
import warnings
import shutil
import re
from unicodedata import normalize
import joblib
import random

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- INICIO DE LA CONFIGURACIÓN DE GPU ---
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        logical_gpus = tf.config.experimental.list_logical_devices('GPU')
        print(f"GPU detectada: {len(gpus)} Physical GPUs, {len(logical_gpus)} Logical GPUs")
        print("TensorFlow utilizará la GPU para el entrenamiento.")
    except RuntimeError as e:
        print(e)
else:
    print("No se detectó ninguna GPU. TensorFlow se ejecutará en la CPU.")
# --- FIN DE LA CONFIGURACIÓN DE GPU ---


# --- Helper Function for Sanitizing TF Names ---
def sanitize_tf_name(name_str):
    s = normalize('NFKD', str(name_str)); s = s.encode('ascii', 'ignore'); s = s.decode('ascii')
    s = re.sub(r'[^a-zA-Z0-9_]', '_', s)
    if s and s[0].isdigit() and not s.isdigit(): s = "n_" + s
    if not s: s = "sanitized_default_name"
    s = re.sub(r'_+', '_', s)
    if s.startswith('_') and len(s) > 1: s = s[1:]
    if s.endswith('_') and len(s) > 1: s = s[:-1]
    if not s: s = "sanitized_default_name_v2"
    return s

# --- 0. Carga de Datos Originales ---
try:
    file_path_load = os.path.join('CSVs', 'Air_Quality.csv')
    df_air_quality_raw = pd.read_csv(file_path_load)
    print(f"Datos originales cargados desde: {file_path_load}")
except FileNotFoundError:
    print("Error: Air_Quality.csv no encontrado.")
    exit()

df_air_quality_raw['Date'] = pd.to_datetime(df_air_quality_raw['Date'], utc=True)
if 'CO2' in df_air_quality_raw.columns:
    df_air_quality_raw = df_air_quality_raw.drop(columns=['CO2'])
df_air_quality_raw['City'] = df_air_quality_raw['City'].replace('New_York', 'New York')

# --- 1. Conversión de Zona Horaria y Extracción de Características de Tiempo ---
print("\nConvirtiendo fechas de UTC a la hora local de cada ciudad y extrayendo características de tiempo...")
city_timezones = {'Brasilia': 'America/Sao_Paulo', 'Cairo': 'Africa/Cairo', 'Dubai': 'Asia/Dubai','London': 'Europe/London', 'New York': 'America/New_York', 'Sydney': 'Australia/Sydney'}
df_parts = []
TIME_FEATURES = ['Hour', 'DayOfWeek', 'Month']
for city, tz in city_timezones.items():
    df_city_part = df_air_quality_raw[df_air_quality_raw['City'] == city].copy()
    if not df_city_part.empty:
        df_city_part['Date'] = df_city_part['Date'].dt.tz_convert(tz)
        df_city_part['Hour'] = df_city_part['Date'].dt.hour
        df_city_part['DayOfWeek'] = df_city_part['Date'].dt.dayofweek
        df_city_part['Month'] = df_city_part['Date'].dt.month
        df_parts.append(df_city_part)
if not df_parts: print("Advertencia: No se procesaron datos."); df_air_quality_full_original = pd.DataFrame(columns=list(df_air_quality_raw.columns) + TIME_FEATURES)
else: df_air_quality_full_original = pd.concat(df_parts).sort_values(by=['City', 'Date']).reset_index(drop=True)
print("Conversión de zona horaria y extracción de características de tiempo completada.")

# --- Parámetros ---
TARGET_CITY_VAE = None
ALL_TARGET_COLS_VAE = ['CO', 'NO2', 'SO2', 'O3', 'PM2.5', 'PM10', 'AQI']
N_LAGS_VAE = 60
LATENT_DIM_VAE, ENCODER_NEURONS_L1_VAE, DECODER_NEURONS_L1_VAE = 16, 64, 64
VAE_INITIAL_LR, KL_LOSS_WEIGHT = 5e-4, 0.00005
VAE_BATCH_SIZE = 64
WFV_INITIAL_TRAIN_POINTS_MIN_RATIO, WFV_VALIDATION_POINTS_MIN_RATIO, WFV_STEP_POINTS_MIN_RATIO = 0.2, 0.1, 0.1
WFV_EPOCHS_PER_FOLD, WFV_ES_PATIENCE, WFV_LR_PATIENCE = 2, 15, 7
MAX_WFV_FOLDS = 2
N_REPETITIONS_FOR_EVAL = 2
NAN_SIMULATION_SCENARIOS = [{'name': name} for name in ['RandomRows_20percent', 'BlockRows_12h_20percent', 'BlockRows_24h_20percent', 'BlockRows_36h_20percent', 'BlockRows_48h_20percent', 'BlockRows_60h_20percent']]

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
SIMULATED_DATA_ROOT_DIR = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'simulados')
LSTM_VAE_METHOD_NAME = 'LSTM_VAE_full'
IMPUTATIONS_LSTM_VAE_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones', LSTM_VAE_METHOD_NAME)
METRICS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'metricas')
IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'CSVs_unified')
MODELS_LSTM_VAE_OUTPUT_DIR = os.path.join(IMPUTATIONS_LSTM_VAE_BASE_DIR, 'model_weights')
os.makedirs(METRICS_LSTM_VAE_OUTPUT_DIR, exist_ok=True); os.makedirs(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, exist_ok=True); os.makedirs(MODELS_LSTM_VAE_OUTPUT_DIR, exist_ok=True)

# --- Funciones VAE y Auxiliares (sin cambios) ---
def sampling(args):
    z_mean, z_log_var = args; batch = tf.shape(z_mean)[0]; dim = tf.shape(z_mean)[1]
    epsilon = tf.random.normal(shape=(batch, dim)); return z_mean + K.exp(0.5 * z_log_var) * epsilon

class VAE(Model):
    def __init__(self, encoder, decoder, kl_loss_weight, **kwargs):
        super().__init__(**kwargs)
        self.encoder, self.decoder, self.kl_loss_weight = encoder, decoder, kl_loss_weight
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name="kl_loss")
        self.mse_loss_fn = keras.losses.MeanSquaredError()
    @property
    def metrics(self): return [self.total_loss_tracker, self.reconstruction_loss_tracker, self.kl_loss_tracker]
    def call(self, inputs): z_mean, _, z = self.encoder(inputs); return self.decoder(z)
    def train_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(x); reconstruction = self.decoder(z)
            reconstruction_loss = self.mse_loss_fn(x[:, :, :1], reconstruction)
            kl_loss = -0.5 * K.mean(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var))
            total_loss = reconstruction_loss + self.kl_loss_weight * kl_loss
        grads = tape.gradient(total_loss, self.trainable_weights); self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.total_loss_tracker.update_state(total_loss); self.reconstruction_loss_tracker.update_state(reconstruction_loss); self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}
    def test_step(self, data):
        x = data[0] if isinstance(data, tuple) else data
        z_mean, z_log_var, z = self.encoder(x); reconstruction = self.decoder(z)
        reconstruction_loss = self.mse_loss_fn(x[:, :, :1], reconstruction)
        kl_loss = -0.5 * K.mean(1 + z_log_var - K.square(z_mean) - K.exp(z_log_var))
        total_loss = reconstruction_loss + self.kl_loss_weight * kl_loss
        self.total_loss_tracker.update_state(total_loss); self.reconstruction_loss_tracker.update_state(reconstruction_loss); self.kl_loss_tracker.update_state(kl_loss)
        return {m.name: m.result() for m in self.metrics}

def create_vae_model_architecture(n_lags, n_features, latent_dim, enc_neurons, dec_neurons, var_name="var"):
    prefix = sanitize_tf_name(var_name)
    enc_in = layers.Input(shape=(n_lags, n_features), name=f"{prefix}_enc_in"); enc_lstm = layers.LSTM(enc_neurons, name=f"{prefix}_enc_lstm")(enc_in)
    z_m, z_lv = layers.Dense(latent_dim, name=f"{prefix}_z_mean")(enc_lstm), layers.Dense(latent_dim, name=f"{prefix}_z_log_var")(enc_lstm)
    z = layers.Lambda(sampling, name=f"{prefix}_z_samp")([z_m, z_lv]); encoder = Model(enc_in, [z_m, z_lv, z], name=f"{prefix}_encoder")
    dec_in = layers.Input(shape=(latent_dim,), name=f"{prefix}_dec_in"); dec_rep = layers.RepeatVector(n_lags, name=f"{prefix}_dec_rep")(dec_in)
    dec_lstm = layers.LSTM(dec_neurons, return_sequences=True, name=f"{prefix}_dec_lstm")(dec_rep)
    dec_out = layers.TimeDistributed(layers.Dense(1), name=f"{prefix}_dec_out")(dec_lstm); decoder = Model(dec_in, dec_out, name=f"{prefix}_decoder")
    return encoder, decoder

def create_sequences_for_vae_training(target_1d, exog_df, n_lags):
    X = []; n_exog = exog_df.shape[1] if exog_df is not None else 0
    for i in range(len(target_1d) - n_lags + 1):
        target_win = target_1d[i : i + n_lags].reshape(-1, 1)
        if n_exog > 0: X.append(np.concatenate((target_win, exog_df.iloc[i : i + n_lags].values), axis=1))
        else: X.append(target_win)
    return np.array(X) if X else np.array([]).reshape(0, n_lags, 1 + n_exog)

def impute_segment_with_lstm_vae_batch(target_series, exog_df_scaled, vae, scaler, n_lags, fallback_val):
    imputed = pd.Series(target_series.copy().flatten()); nan_indices = imputed[imputed.isnull()].index
    if not nan_indices.any(): return imputed.values.reshape(-1, 1)
    temp_filled = imputed.ffill().bfill().fillna(fallback_val)
    sequences, indices = [], []
    for idx in nan_indices:
        if idx < n_lags: continue
        start, end = idx - n_lags, idx
        target_win = scaler.transform(temp_filled.iloc[start:end].values.reshape(-1, 1))
        if exog_df_scaled is not None: seq = np.concatenate((target_win, exog_df_scaled.iloc[start:end].values), axis=1)
        else: seq = target_win
        sequences.append(seq); indices.append(idx)
    if not sequences: return imputed.ffill().bfill().fillna(fallback_val).values.reshape(-1, 1)
    reconstructions = vae.predict(np.array(sequences), batch_size=512, verbose=0)
    imputed_scaled = reconstructions[:, -1, 0]; imputed_unscaled = scaler.inverse_transform(imputed_scaled.reshape(-1, 1)).flatten()
    imputed.iloc[indices] = imputed_unscaled
    return imputed.ffill().bfill().fillna(fallback_val).values.reshape(-1, 1)

def calculate_metrics_with_nmae_nrmse(y_true_df, y_pred_df, nan_mask_df, cols_to_eval):
    res = {}
    y_true, y_pred, mask = y_true_df.reset_index(drop=True), y_pred_df.reset_index(drop=True), nan_mask_df.reset_index(drop=True)
    for col in cols_to_eval:
        if col not in y_true.columns or col not in y_pred.columns: continue
        res[col] = {'MAE': np.nan, 'RMSE': np.nan, 'NRMAE_range': np.nan, 'NRMSE_range': np.nan}
        true_vals, pred_vals, m = y_true[col], y_pred[col], mask[col].astype(bool)
        if m.sum() == 0: continue
        mae, rmse = mean_absolute_error(true_vals[m], pred_vals[m]), np.sqrt(mean_squared_error(true_vals[m], pred_vals[m]))
        data_range = y_true[col].max() - y_true[col].min()
        res[col].update({'MAE': mae, 'RMSE': rmse, 'NRMAE_range': mae / data_range if data_range > 1e-9 else np.nan, 'NRMSE_range': rmse / data_range if data_range > 1e-9 else np.nan})
    return res

# --- Script Principal ---
all_detailed_metrics_lstm_vae_results = []
script_start_time_lstm_vae = time.time()
cities_to_process_vae = [TARGET_CITY_VAE] if TARGET_CITY_VAE else (df_air_quality_full_original['City'].unique() if not df_air_quality_full_original.empty else [])

for city in cities_to_process_vae:
    print(f"\n=== PROCESANDO CIUDAD: {city} ==="); df_city_clean = df_air_quality_full_original[df_air_quality_full_original['City'] == city].copy()
    if df_city_clean.empty: continue
    
    trained_vaes, scalers_target = {}, {}; df_exog_base = df_city_clean[TIME_FEATURES].ffill().bfill()
    if df_exog_base.isnull().any().any(): continue
    
    scaler_exog = MinMaxScaler(); scaled_exog_df = pd.DataFrame(scaler_exog.fit_transform(df_exog_base), columns=TIME_FEATURES)
    n_features = 1 + scaled_exog_df.shape[1]
    
    city_model_dir = os.path.join(MODELS_LSTM_VAE_OUTPUT_DIR, sanitize_tf_name(city)); os.makedirs(city_model_dir, exist_ok=True)
    joblib.dump(scaler_exog, os.path.join(city_model_dir, f'scaler_exog_{sanitize_tf_name(city)}.joblib'))

    for var in ALL_TARGET_COLS_VAE:
        K.clear_session(); tf.random.set_seed(42); np.random.seed(42); random.seed(42)
        print(f"\n--- Preparando VAE para VARIABLE: {var} (Ciudad: {city}) ---")
        
        series_clean = df_city_clean[var].ffill().bfill();
        if series_clean.isnull().any(): continue
        
        scaler_target = MinMaxScaler(); scaled_target_full = scaler_target.fit_transform(series_clean.values.reshape(-1, 1))
        scalers_target[var] = scaler_target
        
        encoder, decoder = create_vae_model_architecture(N_LAGS_VAE, n_features, LATENT_DIM_VAE, ENCODER_NEURONS_L1_VAE, DECODER_NEURONS_L1_VAE, var)
        
        print(f"  Iniciando WFV para VAE de {var}...")
        total_points = len(scaled_target_full); init_train_len = int(total_points * WFV_INITIAL_TRAIN_POINTS_MIN_RATIO)
        val_len = int(total_points * WFV_VALIDATION_POINTS_MIN_RATIO); step_len = max(1, int(total_points * WFV_STEP_POINTS_MIN_RATIO))
        if total_points < init_train_len + val_len + N_LAGS_VAE: print("  Datos insuficientes para WFV."); trained_vaes[var] = None; continue

        current_train_end, fold, best_val_loss = init_train_len, 0, float('inf')
        model_weights_path = os.path.join(city_model_dir, f'vae_weights_best_{sanitize_tf_name(var)}.weights.h5')

        while (current_train_end + val_len <= total_points) and (fold < MAX_WFV_FOLDS):
            fold += 1
            train_target = scaled_target_full[:current_train_end]; val_target = scaled_target_full[current_train_end : current_train_end + val_len]
            train_exog, val_exog = scaled_exog_df.iloc[:current_train_end], scaled_exog_df.iloc[current_train_end : current_train_end + val_len]
            print(f"    WFV Fold {fold} ({var}): Train {len(train_target)}, Val {len(val_target)}")
            
            X_train = create_sequences_for_vae_training(train_target.flatten(), train_exog, N_LAGS_VAE)
            X_val = create_sequences_for_vae_training(val_target.flatten(), val_exog, N_LAGS_VAE)
            if X_train.shape[0] < VAE_BATCH_SIZE or X_val.shape[0] == 0: current_train_end += step_len; continue
            
            vae_model = VAE(encoder, decoder, KL_LOSS_WEIGHT)
            # CORRECCIÓN: Usar tf.keras.optimizers.Adam
            vae_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=VAE_INITIAL_LR), loss=None)
            if not vae_model.built: vae_model(tf.zeros((1, N_LAGS_VAE, n_features)))
            if fold > 1 and os.path.exists(model_weights_path):
                try: vae_model.load_weights(model_weights_path)
                except Exception as e: print(f"      Error cargando pesos en fold {fold}: {e}")

            callbacks = [keras.callbacks.EarlyStopping(monitor='val_total_loss', patience=WFV_ES_PATIENCE, restore_best_weights=True),
                         keras.callbacks.ReduceLROnPlateau(monitor='val_total_loss', patience=WFV_LR_PATIENCE, factor=0.5)]
            
            history = vae_model.fit(X_train, epochs=WFV_EPOCHS_PER_FOLD, batch_size=VAE_BATCH_SIZE, validation_data=(X_val), callbacks=callbacks, verbose=1)
            
            current_val_loss = min(history.history.get('val_total_loss', [float('inf')]))

            print(f"      Fold {fold} completado. Val Total Loss: {current_val_loss:.4f}")
            if current_val_loss < best_val_loss:
                best_val_loss = current_val_loss; print(f"      NUEVO MEJOR VAE ({var}, val_loss: {best_val_loss:.4f}). Guardando pesos.")
                vae_model.save_weights(model_weights_path)
            current_train_end += step_len; K.clear_session()
        
        final_model = VAE(encoder, decoder, KL_LOSS_WEIGHT)
        # CORRECCIÓN: Usar tf.keras.optimizers.Adam
        final_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=VAE_INITIAL_LR), loss=None)
        if not final_model.built: final_model(tf.zeros((1, N_LAGS_VAE, n_features)))
        if os.path.exists(model_weights_path):
            try: final_model.load_weights(model_weights_path); print(f"  Modelo VAE final para {var} cargado.")
            except Exception as e: print(f"  Error cargando pesos finales VAE {var}: {e}."); final_model = None
        else: print(f"  Pesos VAE no encontrados para {var}."); final_model = None
        trained_vaes[var] = final_model

    # --- Imputation Phase ---
    print(f"\n--- Iniciando Imputación para {city} ---")
    city_name_safe = sanitize_tf_name(city)
    for scenario in NAN_SIMULATION_SCENARIOS:
        scenario_name_safe = sanitize_tf_name(scenario['name'])
        print(f"  Procesando Escenario: {scenario_name_safe}")
        imputed_scenario_dir = os.path.join(IMPUTED_CSVS_LSTM_VAE_OUTPUT_DIR, city_name_safe, scenario_name_safe); os.makedirs(imputed_scenario_dir, exist_ok=True)
        
        for rep in range(1, N_REPETITIONS_FOR_EVAL + 1):
            sim_dir = os.path.join(SIMULATED_DATA_ROOT_DIR, city_name_safe, scenario['name'])
            sim_files = glob.glob(os.path.join(sim_dir, f"{city_name_safe}_{scenario['name']}_rep{rep}_*_with_nans.csv"))
            if not sim_files: continue
            
            df_with_nans = pd.read_csv(sim_files[0])
            df_with_nans['Date'] = pd.to_datetime(df_with_nans['Date'], utc=True)
            df_imputed = df_with_nans.copy()
            nan_mask_eval = df_with_nans[ALL_TARGET_COLS_VAE].isnull() & ~df_city_clean[ALL_TARGET_COLS_VAE].reset_index(drop=True).isnull()
            
            exog_for_impute = df_with_nans.reindex(columns=TIME_FEATURES).ffill().bfill()
            scaled_exog_for_impute = pd.DataFrame(scaler_exog.transform(exog_for_impute), columns=TIME_FEATURES) if not exog_for_impute.empty else None

            for var in ALL_TARGET_COLS_VAE:
                vae, scaler_t = trained_vaes.get(var), scalers_target.get(var)
                if not vae or not scaler_t: df_imputed[var] = df_imputed[var].interpolate().ffill().bfill(); continue
                imputed_values = impute_segment_with_lstm_vae_batch(df_with_nans[var].values, scaled_exog_for_impute, vae, scaler_t, N_LAGS_VAE, df_city_clean[var].mean())
                df_imputed[var] = imputed_values.flatten()
            
            imputed_filename = os.path.basename(sim_files[0]).replace("_with_nans.csv", "_imputed_vae_ALL.csv")
            df_imputed.to_csv(os.path.join(imputed_scenario_dir, imputed_filename), index=False)
            print(f"      Archivo imputado guardado: {imputed_filename}")

            metrics = calculate_metrics_with_nmae_nrmse(df_city_clean.reset_index(drop=True), df_imputed, nan_mask_eval, ALL_TARGET_COLS_VAE)
            for var_name, scores in metrics.items():
                all_detailed_metrics_lstm_vae_results.append({'City': city, 'Scenario': scenario_name_safe, 'Repetition': rep, 'Seed': 'N/A', 'Imputation_Method': LSTM_VAE_METHOD_NAME, 'Variable': var_name, **scores})

if all_detailed_metrics_lstm_vae_results:
    df_metrics = pd.DataFrame(all_detailed_metrics_lstm_vae_results)
    sufijo = f"_{sanitize_tf_name(TARGET_CITY_VAE)}" if TARGET_CITY_VAE else "_ALL_CITIES"
    df_metrics.to_csv(os.path.join(METRICS_LSTM_VAE_OUTPUT_DIR, f'ALL_metrics_detailed_{LSTM_VAE_METHOD_NAME}{sufijo}.csv'), index=False)
    print(f"\nMétricas detalladas guardadas.")

print(f"\nProceso LSTM-VAE completado. Tiempo: {(time.time() - script_start_time_lstm_vae) / 60:.2f} min.")

GPU detectada: 1 Physical GPUs, 1 Logical GPUs
TensorFlow utilizará la GPU para el entrenamiento.

ATENCIÓN: Se procesará únicamente la ciudad: Brasilia
ATENCIÓN: Se procesarán todos los escenarios.

--- Procesando datos de: Baseline_Cleaned ---
  Procesando ciudad (Baseline): Brasilia
      -> Iniciando validación cruzada con 5 fold(s)...
        -> Fold 1/5...
          -> Ajustando modelo LSTM...
Epoch 1/50
23/23 [==============================] - 4s 54ms/step - loss: 0.0636 - val_loss: 0.0320
Epoch 2/50
23/23 [==============================] - 0s 21ms/step - loss: 0.0354 - val_loss: 0.0161
Epoch 3/50
23/23 [==============================] - 0s 21ms/step - loss: 0.0222 - val_loss: 0.0120
Epoch 4/50
23/23 [==============================] - 0s 20ms/step - loss: 0.0183 - val_loss: 0.0102
Epoch 5/50
23/23 [==============================] - 0s 20ms/step - loss: 0.0161 - val_loss: 0.0087
Epoch 6/50
23/23 [==============================] - 0s 21ms/step - loss: 0.0130 - val_loss: 0.0067
Epo

MOMENT_PRE UNIVARIADO

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
from momentfm import MOMENTPipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tqdm import tqdm
import time
import glob
import warnings
import traceback

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# --- 0. Configuración Inicial y Parámetros de Predicción ---
TARGET_VARIABLE = 'AQI'
MODEL_NAME = "AutonLab/MOMENT-1-large"
CONTEXT_LENGTH = 512
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
MAX_HORIZON = max(PREDICTION_HORIZONS)

# Parámetros de Ejecución
BATCH_SIZE_PREDICT = 64 # Podemos usar un batch size más grande en CPU

# --- Filtros para Pruebas Rápidas ---
TARGET_CITY_FILTER = "Brasilia"
N_REPETITIONS_TO_PROCESS = 1

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')

PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_PRE_Pred_Univar' # Nuevo nombre para reflejar el enfoque
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)


# --- Dataset para Predicción Univariada ---
class MomentPredictionDataset(Dataset):
    def __init__(self, data_array, context_length, max_horizon):
        self.data_array = data_array
        self.context_length = context_length
        self.max_horizon = max_horizon
        self.n_samples = max(0, len(data_array) - context_length - max_horizon + 1)
    def __len__(self):
        return self.n_samples
    def __getitem__(self, idx):
        context = self.data_array[idx : idx + self.context_length]
        ground_truth_forecast = self.data_array[idx + self.context_length : idx + self.context_length + self.max_horizon]
        return {'context': torch.tensor(context, dtype=torch.float32), 'ground_truth_forecast': torch.tensor(ground_truth_forecast, dtype=torch.float32)}

# --- Funciones Auxiliares de Métrica ---
def calculate_mase(y_true, y_pred, y_train_for_naive):
    y_train_no_nan = y_train_for_naive.dropna()
    if len(y_train_no_nan) <= 1: return np.nan
    mae_naive = np.mean(np.abs(np.diff(y_train_no_nan.values)))
    if mae_naive < 1e-9: return np.nan
    return np.mean(np.abs(y_true - y_pred)) / mae_naive

# --- Función Principal de Predicción (simplificada a train/test split) ---
def predict_and_evaluate_moment_univar(df_city_data, target_col, model_pipeline, device, context_length, horizons, batch_size):
    series_data = df_city_data[target_col].values
    
    # Usar una división simple train/test en lugar de CV
    test_size = int(len(series_data) * 0.2)
    min_len_test = context_length + max(horizons)
    if test_size < min_len_test:
        print(f"      Datos insuficientes para la predicción. Test size ({test_size}) < min requerido ({min_len_test}).")
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
        
    train_data = series_data[:-test_size]
    test_data = series_data[-test_size:]
    
    prediction_dataset = MomentPredictionDataset(test_data, context_length, max(horizons))
    if len(prediction_dataset) == 0:
        print("      No se pudieron crear secuencias de predicción."); return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
        
    prediction_loader = DataLoader(prediction_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    all_predictions, all_ground_truths = [], []
    
    model_pipeline.eval()
    with torch.no_grad():
        for batch in tqdm(prediction_loader, desc="      Prediciendo", leave=False):
            # La entrada es (batch, 1, context_len)
            context_batch = batch['context'].unsqueeze(1).to(device)
            try:
                outputs = model_pipeline(x_enc=context_batch, forecast_len=max(horizons))
                forecasts = outputs.forecast.cpu().squeeze(1)
                all_predictions.append(forecasts)
                all_ground_truths.append(batch['ground_truth_forecast'])
            except Exception as e:
                print(f"        Error en el forward pass del modelo: {e}"); continue
    
    if not all_predictions: return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}

    predictions_np = torch.cat(all_predictions).numpy()
    ground_truths_np = torch.cat(all_ground_truths).numpy()
    results = {}
    for h in horizons:
        y_pred, y_true = predictions_np[:, h-1], ground_truths_np[:, h-1]
        results[h] = {
            'MAE': mean_absolute_error(y_true, y_pred),
            'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
            'MASE': calculate_mase(y_true, y_pred, pd.Series(train_data))
        }
    return results

# --- Script Principal ---
all_prediction_metrics_detailed = []
script_start_time = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"--- Predicción con MOMENT (Pre-entrenado, Univariado) ---"); print(f"Usando dispositivo: {device}")
try:
    moment_predictor = MOMENTPipeline.from_pretrained(
        MODEL_NAME, 
        model_kwargs={'task_name': 'forecasting', 'forecast_horizon': MAX_HORIZON}
    )
    moment_predictor.init(); moment_predictor = moment_predictor.to(device)
    print("Modelo MOMENT (Pre-entrenado) para predicción cargado e inicializado.")
except Exception as e:
    print(f"Error crítico cargando modelo de predicción: {e}"); traceback.print_exc(); exit()

# --- 1. Procesar Baseline ---
try:
    df_original = pd.read_csv(ORIGINAL_DATA_FILE, parse_dates=['Date'])
    cities_to_process = [TARGET_CITY_FILTER] if TARGET_CITY_FILTER else df_original['City'].unique()
    print("\n--- Procesando Baseline ---")
    for city in cities_to_process:
        if city not in df_original['City'].unique(): continue
        print(f"  Ciudad: {city}")
        df_city_data = df_original[df_original['City'] == city].copy()
        df_city_data[TARGET_VARIABLE] = df_city_data[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
        if df_city_data.empty or df_city_data[TARGET_VARIABLE].isnull().any(): continue
        
        horizon_metrics = predict_and_evaluate_moment_univar(df_city_data, TARGET_VARIABLE, moment_predictor, device, CONTEXT_LENGTH, PREDICTION_HORIZONS, BATCH_SIZE_PREDICT)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': 'Baseline_Interpolated', 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})
except FileNotFoundError:
    print(f"Archivo original no encontrado: {ORIGINAL_DATA_FILE}. Omitiendo Baseline.")

# --- 2. Procesar Datos Imputados ---
print("\n--- Procesando Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Método de Imputación: {method_name} ---")
    city_folders = [c for c in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, c)) and (TARGET_CITY_FILTER is None or c == TARGET_CITY_FILTER.replace(" ", "_"))]

    for city_folder in city_folders:
        city_path, city_name = os.path.join(imputed_csvs_root, city_folder), city_folder.replace("_", " ")
        print(f"  Ciudad: {city_name}")

        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")
            
            files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
            if not files_to_process: continue
            
            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                rep_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                print(f"      -> Procesando archivo: {file_name}")

                try:
                    df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                    if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                    if df_rep.empty or df_rep[TARGET_VARIABLE].isnull().any(): continue
                except Exception as e: print(f"        Error cargando {file_name}: {e}"); continue
                
                horizon_metrics = predict_and_evaluate_moment_univar(df_rep, TARGET_VARIABLE, moment_predictor, device, CONTEXT_LENGTH, PREDICTION_HORIZONS, BATCH_SIZE_PREDICT)
                for h, metrics in horizon_metrics.items():
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

# --- Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_path, index=False)
    print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")

    summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")

    global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")

    final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    final_summary.to_csv(final_summary_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
    print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary)

script_end_time = time.time()
print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time)/60:.2f} minutos.")

MOMENT_FT UNIVARIADO

In [20]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import traceback
import time
import glob
import warnings
import joblib
import sys

warnings.filterwarnings("ignore")

original_stdout = sys.stdout
original_stderr = sys.stderr
log_file = None
log_file_path = ""

# --- 1. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
MODEL_NAME = "AutonLab/MOMENT-1-large"
CONTEXT_LENGTH = 512
# SOLUCIÓN: Restaurar los horizontes de predicción deseados
PREDICTION_HORIZONS = [12, 24, 36, 48, 60] 
MAX_HORIZON = max(PREDICTION_HORIZONS)

# --- Parámetros de Fine-Tuning ---
FINETUNE_EPOCHS = 10
BATCH_SIZE_FINETUNE = 32
LR = 1e-5
FINETUNE_TRAIN_SPLIT_RATIO = 0.7
ES_PATIENCE_FT = 15
MIN_DELTA_FT = 1e-6

# --- Parámetros de Predicción ---
N_CV_SPLITS_PREDICT = 5

# --- Filtros para Pruebas ---
TARGET_CITY_FILTER = "Brasilia"
N_REPETITIONS_TO_PROCESS = 5

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_FT_Pred_Univar'
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
MODELS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'modelos')
SCALERS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'scalers')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SCALERS_DIR, exist_ok=True)

# --- 2. Clases y Funciones ---
class MomentForecastingDataset(Dataset):
    def __init__(self, data_array, context_length, max_horizon):
        self.data_array = data_array
        self.context_length = context_length
        self.max_horizon = max_horizon
        self.n_samples = max(0, len(data_array) - context_length - max_horizon + 1)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        context = self.data_array[idx : idx + self.context_length]
        ground_truth_forecast = self.data_array[idx + self.context_length : idx + self.context_length + self.max_horizon]
        return {'context': torch.tensor(context, dtype=torch.float32), 'ground_truth_forecast': torch.tensor(ground_truth_forecast, dtype=torch.float32)}

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive_on_train = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive_on_train < 1e-9: return np.nan
    mae_forecast = np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred)))
    return mae_forecast / mae_naive_on_train

def train_moment_forecasting_model(model, train_loader, device, epochs, variable_name, model_save_path, max_horizon, learning_rate):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    epochs_no_improve = 0
    print(f"  -> Iniciando fine-tuning para '{variable_name}' en {device.upper()} ({epochs} épocas máx)...")
    for epoch in range(epochs):
        total_epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"    Epoch {epoch+1}/{epochs}", leave=False)
        for batch_data in pbar:
            context = batch_data['context'].unsqueeze(1).to(device)
            ground_truth = batch_data['ground_truth_forecast'].unsqueeze(1).to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=context, forecast_len=max_horizon)
                loss = criterion(outputs.forecast, ground_truth)
                if torch.isnan(loss): continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                total_epoch_loss += loss.item()
                pbar.set_postfix(loss=loss.item())
            except Exception as e: continue
        avg_epoch_loss = total_epoch_loss / len(train_loader) if len(train_loader) > 0 else float('inf')
        print(f"    Epoch {epoch+1} Loss: {avg_epoch_loss:.6f}")
        if avg_epoch_loss < best_loss - MIN_DELTA_FT:
            best_loss = avg_epoch_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= ES_PATIENCE_FT:
            print(f"    -> Early stopping en epoch {epoch+1}.")
            break
    print(f"  -> Fine-tuning para '{variable_name}' completado. Mejor pérdida: {best_loss:.6f}")
    if os.path.exists(model_save_path):
        model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

def predict_and_evaluate_moment_cv(df_city_data, target_col, model_pipeline, device, context_length, horizons, scaler, n_cv_splits):
    model_pipeline.eval()
    series_data = df_city_data[target_col]
    
    min_len = context_length + max(horizons) + n_cv_splits 
    if len(series_data) < min_len:
        print(f"      Datos insuficientes para la predicción con CV. Mínimo requerido: ~{min_len}, disponible: {len(series_data)}.")
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
    
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons}

    print(f"      -> Iniciando evaluación con {n_cv_splits} fold(s) de CV...")
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series_data)):
        train_series = series_data.iloc[train_idx]
        context_unscaled = train_series.tail(context_length).values
        if len(context_unscaled) < context_length: continue

        context_scaled = scaler.transform(context_unscaled.reshape(-1, 1)).flatten()
        context_tensor = torch.tensor(context_scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            try:
                outputs = model_pipeline(x_enc=context_tensor, forecast_len=max(horizons))
                predictions_scaled = outputs.forecast.cpu().squeeze().numpy()
                # SOLUCIÓN: Asegurar que la predicción sea al menos un array 1D
                predictions_scaled = np.atleast_1d(predictions_scaled)
            except Exception as e:
                print(f"        Error en predicción del Fold {fold+1}: {e}"); continue
        
        for h in horizons:
            # Ahora el indexado [h-1] es seguro
            y_pred_h_scaled = predictions_scaled[h-1]
            y_true_h_unscaled = series_data.iloc[test_idx[h-1]]
            y_pred_h_unscaled = scaler.inverse_transform(np.array([[y_pred_h_scaled]]))[0, 0]
            
            mae = np.abs(y_true_h_unscaled - y_pred_h_unscaled)
            rmse = (y_true_h_unscaled - y_pred_h_unscaled)**2
            mase = calculate_mase(np.array([y_true_h_unscaled]), np.array([y_pred_h_unscaled]), train_series)

            accumulated_metrics[h]['maes'].append(mae)
            accumulated_metrics[h]['rmses'].append(rmse)
            if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)

    results = {}
    for h in horizons:
        results[h] = {
            'MAE': np.nanmean(accumulated_metrics[h]['maes']) if accumulated_metrics[h]['maes'] else np.nan,
            'RMSE': np.sqrt(np.nanmean(accumulated_metrics[h]['rmses'])) if accumulated_metrics[h]['rmses'] else np.nan,
            'MASE': np.nanmean(accumulated_metrics[h]['mases']) if accumulated_metrics[h]['mases'] else np.nan
        }
    return results

# --- 3. Script Principal ---
all_prediction_metrics_detailed = []
script_start_time_pred = time.time()
training_device = "cuda" if torch.cuda.is_available() else "cpu"
prediction_device = "cpu"
print(f"--- Predicción con MOMENT (Fine-Tuned) ---\nDispositivo de entrenamiento: {training_device.upper()} | Dispositivo de predicción: {prediction_device.upper()}")

try:
    df_full_original = pd.read_csv(ORIGINAL_DATA_FILE, parse_dates=['Date'])
    if TARGET_CITY_FILTER:
        cities_to_process = [TARGET_CITY_FILTER]
        print(f"\nATENCIÓN: Se procesará únicamente la ciudad: {TARGET_CITY_FILTER}")
    else:
        cities_to_process = df_full_original['City'].unique()
        print("\nATENCIÓN: Se procesarán todas las ciudades encontradas.")
except FileNotFoundError:
    print(f"Error crítico: No se encontró el archivo original en {ORIGINAL_DATA_FILE}. Saliendo."); exit()

finetuned_models, scalers = {}, {}
for city in cities_to_process:
    print(f"\n--- Iniciando Fine-Tuning para la ciudad: {city} ---")
    df_city_clean = df_full_original[df_full_original['City'] == city].copy()
    df_city_clean[TARGET_VARIABLE] = df_city_clean[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
    if df_city_clean.empty or df_city_clean[TARGET_VARIABLE].isnull().any(): print(f"  No hay datos limpios para {city}. Omitiendo."); continue

    scaler = StandardScaler()
    train_size = int(len(df_city_clean) * FINETUNE_TRAIN_SPLIT_RATIO)
    train_data_scaled = scaler.fit_transform(df_city_clean[TARGET_VARIABLE].values[:train_size].reshape(-1, 1)).flatten()
    
    ft_dataset = MomentForecastingDataset(train_data_scaled, CONTEXT_LENGTH, MAX_HORIZON)
    ft_loader = DataLoader(ft_dataset, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)
    
    if len(ft_loader) == 0: print(f"  No se pudieron crear lotes de entrenamiento para {city}. Omitiendo."); continue
    
    try:
        moment_predictor_ft = MOMENTPipeline.from_pretrained(MODEL_NAME, model_kwargs={'task_name': 'forecasting', 'forecast_horizon': MAX_HORIZON})
        moment_predictor_ft.init(); moment_predictor_ft = moment_predictor_ft.to(training_device)
        
        city_safe_name = city.replace(" ", "_")
        model_save_path = os.path.join(MODELS_DIR, f"moment_forecaster_ft_{city_safe_name}.pth")
        scaler_save_path = os.path.join(SCALERS_DIR, f"scaler_{city_safe_name}.joblib")
        
        finetuned_model = train_moment_forecasting_model(moment_predictor_ft, ft_loader, training_device, FINETUNE_EPOCHS, TARGET_VARIABLE, model_save_path, MAX_HORIZON, LR)
        finetuned_models[city] = finetuned_model
        scalers[city] = scaler
        joblib.dump(scaler, scaler_save_path)
    except Exception as e:
        print(f"  Error crítico durante el fine-tuning para {city}: {e}"); traceback.print_exc()

print("\n--- Iniciando Fase de Evaluación ---")
# Baseline
print("\n--- Procesando Baseline (Original Interpolado) ---")
for city in cities_to_process:
    if city in finetuned_models:
        print(f"  Ciudad: {city}")
        df_city_data = df_full_original[df_full_original['City'] == city].copy()
        df_city_data[TARGET_VARIABLE] = df_city_data[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
        
        model_to_eval = finetuned_models[city].to(prediction_device)
        scaler_to_eval = scalers[city]

        horizon_metrics = predict_and_evaluate_moment_cv(df_city_data, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': 'Baseline_Interpolated', 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})

# Datos Imputados
print("\n--- Procesando Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Método de Imputación: {method_name} ---")
    city_folders_to_process = [c.replace("_", " ") for c in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, c))]
    
    for city_name in cities_to_process:
        if city_name not in city_folders_to_process or city_name not in finetuned_models: continue
        print(f"  Ciudad: {city_name}")

        model_to_eval = finetuned_models[city_name].to(prediction_device)
        scaler_to_eval = scalers[city_name]
        
        city_path = os.path.join(imputed_csvs_root, city_name.replace(" ", "_"))
        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")
            
            files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
            if not files_to_process: continue
            
            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                rep_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                print(f"      -> Procesando archivo: {file_name}")

                try:
                    df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                    if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                    if df_rep.empty or df_rep[TARGET_VARIABLE].isnull().any(): continue
                except Exception as e: print(f"        Error cargando {file_name}: {e}"); continue
                
                horizon_metrics = predict_and_evaluate_moment_cv(df_rep, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT)
                for h, metrics in horizon_metrics.items():
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

# --- 4. Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_path, index=False)
    print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")

    summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")

    global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")

    final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    final_summary.to_csv(final_summary_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
    print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary)

script_end_time = time.time()
print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time_pred)/60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
print(f"--- FIN DEL SCRIPT: {time.strftime('%Y-%m-%d %H:%M:%S')} ---")
if log_file and not log_file.closed:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

--- Predicción con MOMENT (Fine-Tuned) ---
Dispositivo de entrenamiento: CUDA | Dispositivo de predicción: CPU

ATENCIÓN: Se procesará únicamente la ciudad: Brasilia

--- Iniciando Fine-Tuning para la ciudad: Brasilia ---
  -> Iniciando fine-tuning para 'AQI' en CUDA (10 épocas máx)...


    Epoch 1 Loss: 0.421515


    Epoch 2 Loss: 0.345047


    Epoch 3 Loss: 0.322024


    Epoch 4 Loss: 0.307889


...

    Epoch 6 Loss: 0.285746


    Epoch 7 Loss: 0.278212


    Epoch 8 Loss: 0.271839


    Epoch 9 Loss: 0.265815


    Epoch 10 Loss: 0.261323
  -> Fine-tuning para 'AQI' completado. Mejor pérdida: 0.261323

--- Iniciando Fase de Evaluación ---

--- Procesando Baseline (Original Interpolado) ---
  Ciudad: Brasilia
      -> Iniciando evaluación con 5 fold(s) de CV...

--- Procesando Datos Imputados ---

--- Método de Imputación: interpolacionLineal ---
  Ciudad: Brasilia
    Escenario: BlockRows_12h_20percent
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep1_seed2510790048_imputed_interpolacionLineal.csv
      -> Iniciando evaluación con 5 fold(s) de CV...
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep2_seed2510854048_imputed_interpolacionLineal.csv
      -> Iniciando evaluación con 5 fold(s) de CV...
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep3_seed2510774176_imputed_interpolacionLineal.csv
      -> Iniciando evaluación con 5 fold(s) de CV...
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep4_seed2510774688_imputed_interpolac

el mejor probablemente, con GPU

In [ ]:
import pandas as pd
import numpy as np
import os
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
from momentfm import MOMENTPipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
import traceback
import time
import glob
import warnings
import joblib
import sys
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# --- 1. Configuración Inicial y Parámetros ---
TARGET_VARIABLE = 'AQI'
MODEL_NAME = "AutonLab/MOMENT-1-large"
CONTEXT_LENGTH = 512
PREDICTION_HORIZONS = [12, 24, 36, 48, 60]
MAX_HORIZON = max(PREDICTION_HORIZONS)

# --- Parámetros de Fine-Tuning ---
FINETUNE_EPOCHS = 2 # Reducido para pruebas rápidas
BATCH_SIZE_FINETUNE = 32
LR = 1e-4
FINETUNE_TRAIN_SPLIT_RATIO = 0.8
ES_PATIENCE_FT = 5
MIN_DELTA_FT = 1e-5

# --- Parámetros de Predicción ---
N_CV_SPLITS_PREDICT = 2 # Reducido para pruebas rápidas

# --- Filtros para Pruebas ---
TARGET_CITY_FILTER = None  # None para procesar todas las ciudades
N_REPETITIONS_TO_PROCESS = 2

# --- Rutas ---
PROJECT_ROOT_DIR = os.getcwd()
ORIGINAL_DATA_FILE = os.path.join(PROJECT_ROOT_DIR, 'CSVs', 'Air_Quality.csv')
IMPUTATIONS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'imputaciones')
PREDICTION_METRICS_BASE_DIR = os.path.join(PROJECT_ROOT_DIR, 'predicciones')
PREDICTION_METHOD_NAME = 'MOMENT_FT_test'
PREDICTION_METRICS_ROOT_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'metricas')
MODELS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'modelos')
SCALERS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'scalers')
PREDICTION_PLOTS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'prediction_plots')
PREDICTION_CSVS_DIR = os.path.join(PREDICTION_METRICS_BASE_DIR, PREDICTION_METHOD_NAME, 'predictions_csvs')
os.makedirs(PREDICTION_METRICS_ROOT_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(SCALERS_DIR, exist_ok=True)
os.makedirs(PREDICTION_PLOTS_DIR, exist_ok=True)
os.makedirs(PREDICTION_CSVS_DIR, exist_ok=True)


# --- 2. Clases y Funciones ---
class MomentForecastingDataset(Dataset):
    def __init__(self, data_array, context_length, max_horizon):
        self.data_array = data_array
        self.context_length, self.max_horizon = context_length, max_horizon
        self.n_samples = max(0, len(data_array) - context_length - max_horizon + 1)
    def __len__(self): return self.n_samples
    def __getitem__(self, idx):
        context = self.data_array[idx : idx + self.context_length]
        target = self.data_array[idx + self.context_length : idx + self.context_length + self.max_horizon]
        return {'context': torch.tensor(context, dtype=torch.float32), 'ground_truth_forecast': torch.tensor(target, dtype=torch.float32)}

def calculate_mase(y_true, y_pred, y_train_for_naive):
    if len(y_train_for_naive) <= 1: return np.nan
    mae_naive = np.mean(np.abs(np.diff(np.asarray(y_train_for_naive))))
    if mae_naive < 1e-9: return np.nan
    return np.mean(np.abs(y_true - y_pred)) / mae_naive

def train_moment_forecasting_model(model, train_loader, device, epochs, variable_name, model_save_path, max_horizon, learning_rate):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    best_loss = float('inf')
    epochs_no_improve = 0
    print(f"  -> Iniciando fine-tuning (linear probing) para '{variable_name}' en {device.upper()} ({epochs} épocas máx)...")
    for epoch in range(epochs):
        total_epoch_loss = 0
        pbar = tqdm(train_loader, desc=f"    Epoch {epoch+1}/{epochs}", leave=False)
        for batch_data in pbar:
            context, ground_truth = batch_data['context'].unsqueeze(1).to(device), batch_data['ground_truth_forecast'].unsqueeze(1).to(device)
            optimizer.zero_grad()
            try:
                outputs = model(x_enc=context, forecast_len=max_horizon)
                loss = criterion(outputs.forecast, ground_truth)
                if torch.isnan(loss): continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                total_epoch_loss += loss.item()
                pbar.set_postfix(loss=loss.item())
            except Exception: continue
        avg_epoch_loss = total_epoch_loss / len(train_loader) if len(train_loader) > 0 else float('inf')
        print(f"    Epoch {epoch+1} Loss: {avg_epoch_loss:.6f}")
        if avg_epoch_loss < best_loss - MIN_DELTA_FT:
            best_loss = avg_epoch_loss; epochs_no_improve = 0
            torch.save(model.state_dict(), model_save_path)
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= ES_PATIENCE_FT:
            print(f"    -> Early stopping en epoch {epoch+1}."); break
    print(f"  -> Fine-tuning completado. Mejor pérdida: {best_loss:.6f}")
    if os.path.exists(model_save_path): model.load_state_dict(torch.load(model_save_path, map_location=device))
    return model

def save_prediction_results_to_csv(context_data, ground_truth_data, prediction_data, date_index, file_path):
    # SOLUCIÓN: Usar .values para ignorar el índice de la serie de fechas y crear uno nuevo.
    df_pred = pd.DataFrame({
        'date': date_index.values,
        'context': np.nan,
        'ground_truth': np.nan,
        'prediction': np.nan
    })
    
    context_len, horizon_len = len(context_data), len(ground_truth_data)
    df_pred.loc[:context_len-1, 'context'] = context_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'ground_truth'] = ground_truth_data
    df_pred.loc[context_len:context_len+horizon_len-1, 'prediction'] = prediction_data
    
    df_pred.to_csv(file_path, index=False)

def plot_and_save_forecast(context, ground_truth, prediction, title, file_path):
    plt.figure(figsize=(15, 6))
    x_context = np.arange(len(context))
    x_forecast = np.arange(len(context), len(context) + len(ground_truth))
    plt.plot(x_context, context, label='Contexto (Datos de Entrada)', color='blue')
    plt.plot(x_forecast, ground_truth, label='Valores Reales (Ground Truth)', color='orange', linestyle='--')
    plt.plot(x_forecast, prediction, label='Predicción del Modelo', color='green', linestyle='-.')
    plt.title(title, fontsize=16)
    plt.xlabel('Pasos de Tiempo (Horas)')
    plt.ylabel('Valor de AQI')
    plt.legend(); plt.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout(); plt.savefig(file_path); plt.close()

def predict_and_evaluate_moment_cv(df_city_data, target_col, model_pipeline, device, context_length, horizons, scaler, n_cv_splits, **kwargs):
    model_pipeline.eval()
    series_data = df_city_data[target_col]
    min_len = context_length + max(horizons) + n_cv_splits 
    if len(series_data) < min_len:
        print(f"      Datos insuficientes para la predicción con CV. Mínimo requerido: ~{min_len}, disponible: {len(series_data)}.")
        return {h: {'MAE': np.nan, 'RMSE': np.nan, 'MASE': np.nan} for h in horizons}
    
    tscv = TimeSeriesSplit(n_splits=n_cv_splits, test_size=max(horizons))
    accumulated_metrics = {h: {'maes': [], 'rmses': [], 'mases': []} for h in horizons}
    print(f"      -> Iniciando evaluación con {n_cv_splits} fold(s) de CV...")
    last_fold_data = {}
    for fold, (train_idx, test_idx) in enumerate(tscv.split(series_data)):
        train_series = series_data.iloc[train_idx]
        context_unscaled = train_series.tail(context_length).values
        if len(context_unscaled) < context_length: continue
        context_scaled = scaler.transform(context_unscaled.reshape(-1, 1)).flatten()
        context_tensor = torch.tensor(context_scaled, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
        with torch.no_grad():
            try:
                outputs = model_pipeline(x_enc=context_tensor, forecast_len=max(horizons))
                predictions_scaled = np.atleast_1d(outputs.forecast.cpu().squeeze().numpy())
            except Exception as e: print(f"        Error en predicción del Fold {fold+1}: {e}"); continue
        
        ground_truth_unscaled = series_data.iloc[test_idx].values
        predictions_unscaled = scaler.inverse_transform(predictions_scaled.reshape(-1, 1)).flatten()

        for h in horizons:
            y_pred_h, y_true_h = predictions_unscaled[h-1], ground_truth_unscaled[h-1]
            mae, rmse = np.abs(y_true_h - y_pred_h), (y_true_h - y_pred_h)**2
            mase = calculate_mase(np.array([y_true_h]), np.array([y_pred_h]), train_series)
            accumulated_metrics[h]['maes'].append(mae); accumulated_metrics[h]['rmses'].append(rmse)
            if not np.isnan(mase): accumulated_metrics[h]['mases'].append(mase)
        
        if fold == n_cv_splits - 1:
            start_idx, end_idx = train_idx[-context_length], test_idx[-1]
            last_fold_data = {'context': context_unscaled, 'ground_truth': ground_truth_unscaled, 'prediction': predictions_unscaled, 'date_index': df_city_data['Date'].iloc[start_idx:end_idx+1]}

    if last_fold_data:
        plot_and_save_forecast(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], kwargs['plot_title'], kwargs['plot_path'])
        save_prediction_results_to_csv(last_fold_data['context'], last_fold_data['ground_truth'], last_fold_data['prediction'], last_fold_data['date_index'], kwargs['csv_path'])
        print(f"      -> Gráfica y CSV de predicción del último fold guardados.")

    results = {}
    for h in horizons:
        results[h] = {'MAE': np.nanmean(accumulated_metrics[h]['maes']), 'RMSE': np.sqrt(np.nanmean(accumulated_metrics[h]['rmses'])), 'MASE': np.nanmean(accumulated_metrics[h]['mases'])}
    return results

# --- 3. Script Principal ---
all_prediction_metrics_detailed = []
script_start_time_pred = time.time()
training_device = "cuda" if torch.cuda.is_available() else "cpu"
prediction_device = "cpu"
print(f"--- Predicción con MOMENT (Fine-Tuned) ---\nDispositivo de entrenamiento: {training_device.upper()} | Dispositivo de predicción: {prediction_device.upper()}")

try:
    df_full_original = pd.read_csv(ORIGINAL_DATA_FILE, parse_dates=['Date'])
    if TARGET_CITY_FILTER: cities_to_process = [TARGET_CITY_FILTER]
    else: cities_to_process = df_full_original['City'].unique()
    print(f"\nATENCIÓN: Se procesarán las siguientes ciudades: {', '.join(cities_to_process)}")
except FileNotFoundError:
    print(f"Error crítico: No se encontró el archivo original en {ORIGINAL_DATA_FILE}. Saliendo."); exit()

finetuned_models, scalers = {}, {}
for city in cities_to_process:
    print(f"\n--- Iniciando Fine-Tuning para la ciudad: {city} ---")
    df_city_clean = df_full_original[df_full_original['City'] == city].copy()
    df_city_clean[TARGET_VARIABLE] = df_city_clean[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
    if df_city_clean.empty or df_city_clean[TARGET_VARIABLE].isnull().any(): print(f"  No hay datos limpios para {city}. Omitiendo."); continue

    scaler = StandardScaler()
    train_size = int(len(df_city_clean) * FINETUNE_TRAIN_SPLIT_RATIO)
    train_data_scaled = scaler.fit_transform(df_city_clean[TARGET_VARIABLE].values[:train_size].reshape(-1, 1)).flatten()
    ft_dataset = MomentForecastingDataset(train_data_scaled, CONTEXT_LENGTH, MAX_HORIZON)
    ft_loader = DataLoader(ft_dataset, batch_size=BATCH_SIZE_FINETUNE, shuffle=True, num_workers=0)
    
    if len(ft_loader) == 0: print(f"  No se pudieron crear lotes de entrenamiento para {city}. Omitiendo."); continue
    
    try:
        model_kwargs = {'task_name': 'forecasting', 
                        'forecast_horizon': MAX_HORIZON, 
                        'freeze_encoder': True, 
                        'freeze_embedder': True}
        moment_predictor_ft = MOMENTPipeline.from_pretrained(MODEL_NAME, model_kwargs=model_kwargs)
        moment_predictor_ft.init(); moment_predictor_ft = moment_predictor_ft.to(training_device)
        
        city_safe_name = city.replace(" ", "_")
        model_save_path = os.path.join(MODELS_DIR, f"moment_forecaster_ft_{city_safe_name}.pth")
        scaler_save_path = os.path.join(SCALERS_DIR, f"scaler_{city_safe_name}.joblib")
        
        finetuned_model = train_moment_forecasting_model(moment_predictor_ft, ft_loader, training_device, FINETUNE_EPOCHS, TARGET_VARIABLE, model_save_path, MAX_HORIZON, LR)
        finetuned_models[city] = finetuned_model
        scalers[city] = scaler
        joblib.dump(scaler, scaler_save_path)
    except Exception as e:
        print(f"  Error crítico durante el fine-tuning para {city}: {e}"); traceback.print_exc()

print("\n--- Iniciando Fase de Evaluación ---")
# Baseline
print("\n--- Procesando Baseline (Original Interpolado) ---")
for city in cities_to_process:
    if city in finetuned_models:
        print(f"  Ciudad: {city}")
        df_city_data = df_full_original[df_full_original['City'] == city].copy()
        df_city_data[TARGET_VARIABLE] = df_city_data[TARGET_VARIABLE].interpolate(method='linear').ffill().bfill()
        
        model_to_eval, scaler_to_eval = finetuned_models[city].to(prediction_device), scalers[city]
        
        plot_title = f'Predicción Baseline - {city}'
        plot_path = os.path.join(PREDICTION_PLOTS_DIR, f'baseline_{city.replace(" ", "_")}.png')
        csv_path = os.path.join(PREDICTION_CSVS_DIR, f'baseline_{city.replace(" ", "_")}.csv')

        horizon_metrics = predict_and_evaluate_moment_cv(df_city_data, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
        for h, metrics in horizon_metrics.items():
            all_prediction_metrics_detailed.append({'Imputation_Method': 'Baseline_Interpolated', 'City': city, 'Scenario': 'N/A', 'Repetition_File_ID': 'N/A', 'Horizon_h': h, **metrics})

# Datos Imputados
print("\n--- Procesando Datos Imputados ---")
imputation_methods = [d for d in os.listdir(IMPUTATIONS_BASE_DIR) if os.path.isdir(os.path.join(IMPUTATIONS_BASE_DIR, d))]
for method_name in imputation_methods:
    imputed_csvs_root = os.path.join(IMPUTATIONS_BASE_DIR, method_name, 'CSVs')
    if not os.path.exists(imputed_csvs_root): continue
    
    print(f"\n--- Método de Imputación: {method_name} ---")
    city_folders_to_process = [c.replace("_", " ") for c in os.listdir(imputed_csvs_root) if os.path.isdir(os.path.join(imputed_csvs_root, c))]
    
    for city_name in cities_to_process:
        if city_name not in city_folders_to_process or city_name not in finetuned_models: continue
        print(f"  Ciudad: {city_name}")

        model_to_eval, scaler_to_eval = finetuned_models[city_name].to(prediction_device), scalers[city_name]
        
        city_path = os.path.join(imputed_csvs_root, city_name.replace(" ", "_"))
        for scenario_folder in os.listdir(city_path):
            scenario_path = os.path.join(city_path, scenario_folder)
            if not os.path.isdir(scenario_path): continue
            print(f"    Escenario: {scenario_folder}")
            
            files_to_process = sorted(glob.glob(os.path.join(scenario_path, "*.csv")))[:N_REPETITIONS_TO_PROCESS]
            if not files_to_process: continue
            
            for rep_file_path in files_to_process:
                file_name = os.path.basename(rep_file_path)
                rep_id = "_".join(file_name.split('_')[-4:-2]) if "_rep" in file_name and "_seed" in file_name else file_name
                print(f"      -> Procesando archivo: {file_name}")

                try:
                    df_rep = pd.read_csv(rep_file_path, parse_dates=['Date'])
                    if df_rep[TARGET_VARIABLE].isnull().any(): df_rep[TARGET_VARIABLE] = df_rep[TARGET_VARIABLE].ffill().bfill()
                    if df_rep.empty or df_rep[TARGET_VARIABLE].isnull().any(): continue
                except Exception as e: print(f"        Error cargando {file_name}: {e}"); continue
                
                plot_title = f'Predicción - {city_name}\nMétodo: {method_name}\nEscenario: {scenario_folder} - {rep_id}'
                plot_path = os.path.join(PREDICTION_PLOTS_DIR, f'{method_name}_{city_name.replace(" ","_")}_{scenario_folder}_{rep_id}.png')
                csv_path = os.path.join(PREDICTION_CSVS_DIR, f'{method_name}_{city_name.replace(" ","_")}_{scenario_folder}_{rep_id}.csv')
                
                horizon_metrics = predict_and_evaluate_moment_cv(df_rep, TARGET_VARIABLE, model_to_eval, prediction_device, CONTEXT_LENGTH, PREDICTION_HORIZONS, scaler_to_eval, N_CV_SPLITS_PREDICT, plot_title=plot_title, plot_path=plot_path, csv_path=csv_path)
                for h, metrics in horizon_metrics.items():
                    all_prediction_metrics_detailed.append({'Imputation_Method': method_name, 'City': city_name, 'Scenario': scenario_folder, 'Repetition_File_ID': rep_id, 'Horizon_h': h, **metrics})

# --- 4. Guardado de Métricas ---
if all_prediction_metrics_detailed:
    df_all_metrics = pd.DataFrame(all_prediction_metrics_detailed)
    
    detailed_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'ALL_prediction_metrics_detailed_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.to_csv(detailed_path, index=False)
    print(f"\n[OK] Métricas detalladas guardadas en: {detailed_path}")

    summary_city_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'summary_by_city_{PREDICTION_METHOD_NAME}.csv')
    df_all_metrics.groupby(['City', 'Imputation_Method', 'Scenario', 'Horizon_h']).agg(Avg_MAE=('MAE', 'mean'), Avg_RMSE=('RMSE', 'mean'), Avg_MASE=('MASE', 'mean')).reset_index().to_csv(summary_city_path, index=False)
    print(f"[OK] Resumen por ciudad y escenario guardado en: {summary_city_path}")

    global_summary = df_all_metrics.groupby(['Imputation_Method', 'Horizon_h']).agg(Global_Avg_MAE=('MAE', 'mean'), Global_Avg_RMSE=('RMSE', 'mean'), Global_Avg_MASE=('MASE', 'mean')).reset_index()
    global_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'global_summary_{PREDICTION_METHOD_NAME}.csv')
    global_summary.to_csv(global_summary_path, index=False)
    print(f"[OK] Resumen global por método y horizonte guardado en: {global_summary_path}")

    final_summary = global_summary.groupby('Imputation_Method').agg(Final_Avg_MAE=('Global_Avg_MAE', 'mean'), Final_Avg_RMSE=('Global_Avg_RMSE', 'mean'), Final_Avg_MASE=('Global_Avg_MASE', 'mean')).reset_index()
    final_summary_path = os.path.join(PREDICTION_METRICS_ROOT_DIR, f'final_summary_avg_over_horizons_{PREDICTION_METHOD_NAME}.csv')
    final_summary.to_csv(final_summary_path, index=False)
    print(f"\n[OK] Resumen final promediando horizontes guardado en: {final_summary_path}")
    print("\n--- Resumen Final por Método (promedio de todo) ---\n", final_summary)

script_end_time = time.time()
print(f"\nScript completado. Tiempo total: {(script_end_time - script_start_time_pred)/60:.2f} minutos.")

# --- Restaurar stdout y stderr y cerrar archivo de log ---
if log_file and not log_file.closed:
    sys.stdout = original_stdout
    sys.stderr = original_stderr
    log_file.close()
    print(f"Salida de log completada y guardada en: {log_file_path}")

--- Predicción con MOMENT (Fine-Tuned) ---
Dispositivo de entrenamiento: CUDA | Dispositivo de predicción: CPU

ATENCIÓN: Se procesarán las siguientes ciudades: Brasilia

--- Iniciando Fine-Tuning para la ciudad: Brasilia ---
  -> Iniciando fine-tuning (linear probing) para 'AQI' en CUDA (10 épocas máx)...


    Epoch 1 Loss: 0.302178


    Epoch 2 Loss: 0.243723


    Epoch 3 Loss: 0.225638


    Epoch 4 Loss: 0.211726


...

    Epoch 6 Loss: 0.198084


    Epoch 7 Loss: 0.191504


    Epoch 8 Loss: 0.185344


    Epoch 9 Loss: 0.181070


    Epoch 10 Loss: 0.177486
  -> Fine-tuning completado. Mejor pérdida: 0.177486

--- Iniciando Fase de Evaluación ---

--- Procesando Baseline (Original Interpolado) ---
  Ciudad: Brasilia
      -> Iniciando evaluación con 5 fold(s) de CV...
      -> Gráfica y CSV de predicción del último fold guardados.

--- Procesando Datos Imputados ---

--- Método de Imputación: interpolacionLineal ---
  Ciudad: Brasilia
    Escenario: BlockRows_12h_20percent
      -> Procesando archivo: Brasilia_BlockRows_12h_20percent_rep1_seed2510790048_imputed_interpolacionLineal.csv
      -> Iniciando evaluación con 5 fold(s) de CV...
      -> Gráfica y CSV de predicción del último fold guardados.
    Escenario: BlockRows_24h_20percent
      -> Procesando archivo: Brasilia_BlockRows_24h_20percent_rep1_seed837057928_imputed_interpolacionLineal.csv
      -> Iniciando evaluación con 5 fold(s) de CV...
      -> Gráfica y CSV de predicción del último fold guardados.
    Escenario: BlockRows_36h_20percent
      -> 